# Legal-RAG Colab: Chunk -> Embed -> Upsert

Notebook này chỉ giữ pipeline indexing:
1. Chunk văn bản pháp luật.
2. Tạo dense + sparse embedding bằng BGE-M3.
3. In thử 1 payload mẫu.
4. Upsert lên Qdrant.

Không dùng LLM chat trong notebook này.

In [35]:
# try:
#     from google.colab import drive
#     # Lệnh này sẽ bật một popup yêu cầu bạn cấp quyền truy cập Google Drive
#     drive.mount('/content/drive')
#     DRIVE_DESTINATION_DIR = "/content/drive/MyDrive/"
#     print("✅ Đã kết nối thành công với Google Drive.")
# except ImportError:
#     print("⚠️ Không chạy trên môi trường Google Colab. Snapshot sẽ được lưu ở thư mục hiện tại.")
#     DRIVE_DESTINATION_DIR = "./"

In [36]:
# @title Cấu hình runtime + Qdrant + HF

import os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
COLLECTION_NAME = "legal_rag_3sectors_test" # @param {type:"string"}
DOC_LIMIT = 100 # @param {type:"integer"}

if IN_KAGGLE:
    QDRANT_LOCAL_PATH = "/kaggle/working/local_qdrant_data"
elif IN_COLAB:
    QDRANT_LOCAL_PATH = "/content/local_qdrant_data"
else:
    QDRANT_LOCAL_PATH = "./local_qdrant_data"

QDRANT_LOCAL_URL = "http://localhost:6333" # @param {type:"string"}
QDRANT_MODE = "cloud" # @param ["local_path", "local_url", "cloud"]
QDRANT_URL = "YOUR_QDRANT_URL" # @param {type:"string"}
QDRANT_API_KEY = "YOUR_QDRANT_API_KEY" # @param {type:"string"}
HF_API_KEY = "hf_YOUR_HUGGINGFACE_TOKEN" # @param {type:"string"}
NEO4J_URI = "YOUR_NEO4J_URI" # @param {type:"string"}
NEO4J_USERNAME = "7b7f7d06" # @param {type:"string"}
NEO4J_PASSWORD = "YOUR_NEO4J_PASSWORD" # @param {type:"string"}

if QDRANT_MODE == "local_path":
    Path(QDRANT_LOCAL_PATH).mkdir(parents=True, exist_ok=True)

os.environ["QDRANT_MODE"] = QDRANT_MODE
os.environ["QDRANT_LOCAL_PATH"] = QDRANT_LOCAL_PATH
os.environ["QDRANT_LOCAL_URL"] = QDRANT_LOCAL_URL
os.environ["QDRANT_URL"] = QDRANT_URL
os.environ["QDRANT_API_KEY"] = QDRANT_API_KEY
os.environ["DOC_LIMIT"] = str(DOC_LIMIT)
os.environ["HF_TOKEN"] = HF_API_KEY

try:
    if HF_API_KEY:
        from huggingface_hub import login
        login(token=HF_API_KEY, add_to_git_credential=False)
        print("Hugging Face login: OK")
except Exception as exc:
    print(f"Hugging Face login warning: {exc}")

print(f"IN_KAGGLE={IN_KAGGLE} | IN_COLAB={IN_COLAB}")
print(f"QDRANT_MODE={QDRANT_MODE} | COLLECTION={COLLECTION_NAME}")
print(f"DOC_LIMIT={DOC_LIMIT} (0 = all)")



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login: OK
IN_KAGGLE=True | IN_COLAB=True
QDRANT_MODE=cloud | COLLECTION=legal_rag_3sectors_test
DOC_LIMIT=100 (0 = all)


In [37]:
%pip install -q -U neo4j qdrant-client datasets pandas accelerate python-dotenv ipywidgets sentence-transformers fastembed pypdf python-docx "FlagEmbedding==1.3.5" "transformers==4.48.3" huggingface-hub tokenizers

Note: you may need to restart the kernel to use updated packages.


In [38]:
# Setup Qdrant client + BGE-M3 hybrid encoder
import os
from pathlib import Path
from typing import Dict, List, Tuple

import torch
from qdrant_client import QdrantClient, models
from FlagEmbedding import BGEM3FlagModel

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DENSE_MODEL_NAME = os.getenv("LEGAL_DENSE_MODEL", "BAAI/bge-m3")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

QDRANT_MODE = os.getenv("QDRANT_MODE", "cloud").lower()
QDRANT_LOCAL_PATH = os.getenv("QDRANT_LOCAL_PATH", "./local_qdrant_data")
QDRANT_LOCAL_URL = os.getenv("QDRANT_LOCAL_URL", "http://localhost:6333")
QDRANT_CLOUD_URL = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
COLLECTION_NAME = globals().get("COLLECTION_NAME", "legal_rag_docs_5000")


def init_qdrant_client() -> QdrantClient:
    if QDRANT_MODE == "local_path":
        Path(QDRANT_LOCAL_PATH).mkdir(parents=True, exist_ok=True)
        client = QdrantClient(path=QDRANT_LOCAL_PATH)
        client.get_collections()
        print(f"Qdrant connected (local_path): {QDRANT_LOCAL_PATH}")
        return client

    if QDRANT_MODE == "local_url":
        client = QdrantClient(url=QDRANT_LOCAL_URL)
        client.get_collections()
        print(f"Qdrant connected (local_url): {QDRANT_LOCAL_URL}")
        return client

    if not QDRANT_CLOUD_URL:
        raise ValueError("QDRANT_URL đang rỗng trong cloud mode.")

    client = QdrantClient(url=QDRANT_CLOUD_URL, api_key=(QDRANT_API_KEY or None))
    client.get_collections()
    print(f"Qdrant connected (cloud): {QDRANT_CLOUD_URL}")
    return client


class LocalBGEHybridEncoder:
    def __init__(self, model_name: str = "BAAI/bge-m3", device: str = "cpu"):
        self.model = BGEM3FlagModel(model_name, use_fp16=(device == "cuda"), device=device)

    @staticmethod
    def _to_sparse_vector(weights: Dict[str, float]) -> models.SparseVector:
        if not weights:
            return models.SparseVector(indices=[], values=[])
        pairs = [(int(k), float(v)) for k, v in weights.items() if float(v) != 0.0]
        pairs.sort(key=lambda x: x[0])
        return models.SparseVector(indices=[i for i, _ in pairs], values=[v for _, v in pairs])

    def encode_hybrid(self, texts: List[str], batch_size: int = 16) -> Tuple[List[List[float]], List[models.SparseVector]]:
        if isinstance(texts, str):
            texts = [texts]

        out = self.model.encode(
            texts,
            batch_size=(256 if torch.cuda.is_available() else batch_size),
            max_length=2048,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False,
        )

        dense_vecs = out["dense_vecs"]
        dense_list = dense_vecs.tolist() if hasattr(dense_vecs, "tolist") else [list(v) for v in dense_vecs]
        sparse_list = [self._to_sparse_vector(w) for w in out["lexical_weights"]]
        return dense_list, sparse_list


qdrant_client = init_qdrant_client()
hybrid_encoder = LocalBGEHybridEncoder(model_name=DENSE_MODEL_NAME, device=DEVICE)

dense_dim = len(hybrid_encoder.encode_hybrid(["probe"], batch_size=1)[0][0])
print(f"Device={DEVICE} | dense_dim={dense_dim}")

if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "dense": models.VectorParams(size=dense_dim, distance=models.Distance.COSINE, on_disk=True)
        },
        sparse_vectors_config={
            # ĐỔI THÀNH True Ở ĐÂY ĐỂ TRÁNH TRÀN RAM
            "sparse": models.SparseVectorParams(index=models.SparseIndexParams(on_disk=True))
        },
    )
    print(f"Created collection: {COLLECTION_NAME}")
else:
    print(f"Collection exists: {COLLECTION_NAME}")

Qdrant connected (cloud): https://c11d9aa7-751f-4269-8901-575a9e786d36.eu-west-1-0.aws.cloud.qdrant.io:6333


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

initial target device:   0%|          | 0/2 [00:00<?, ?it/s]2026-04-16 08:56:31.787291: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776329791.812895    1413 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776329791.821120    1413 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776329791.844053    1413 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776329791.844090    1413 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776329791.8440

Device=cuda | dense_dim=1024
Collection exists: legal_rag_3sectors_test


In [39]:
import os
import re
from typing import Any, List
import pandas as pd
from datasets import load_dataset
from IPython.display import display

def split_sector_list(value: Any) -> List[str]:
    if pd.isna(value) or not value: return []
    raw_text = ", ".join(str(x) for x in value if str(x).strip()) if isinstance(value, list) else str(value).strip()
    if not raw_text or raw_text.lower() == "nan": return []
    return sorted(set(part.strip() for part in re.split(r"\s*,\s*", raw_text) if part and part.strip()))

def parse_signer_name(signer_str: Any) -> str:
    if pd.isna(signer_str) or not str(signer_str).strip(): return ""
    return str(signer_str).split(":")[0].strip()

def parse_signer_id(signer_str: Any) -> Any:
    if pd.isna(signer_str) or not str(signer_str).strip(): return None
    parts = str(signer_str).split(":")
    if len(parts) > 1 and parts[1].strip().isdigit(): return int(parts[1].strip())
    return None

# --- 1. ĐỌC DANH SÁCH ID TỪ FILE CSV KAGGLE ---
csv_path = "/kaggle/input/datasets/nhn309261/medical-legal-docs-numbers/metadata_the_thao_y_te_100.csv"

# Nếu bạn chạy local hoặc colab không có đường dẫn này, có thể tùy biến lại
if os.path.exists(csv_path):
    print(f"Loading target IDs from CSV: {csv_path}")
    df_csv = pd.read_csv(csv_path)
    target_ids = set(df_csv['id'].astype(str).tolist())
    print(f"-> Found {len(target_ids)} IDs in CSV.\n")
else:
    print(f"⚠️ Không tìm thấy file {csv_path}. Bạn cần upload file csv lên Kaggle.")
    target_ids = set()

# --- 2. LOAD DATA TỪ HUGGING FACE ---
print("Loading dataset from Hugging Face...")
ds_metadata_all = load_dataset("nhn309261/vietnamese-legal-docs", "metadata", split="data")
ds_content_all = load_dataset("nhn309261/vietnamese-legal-docs", "content", split="data")

# --- 3. LỌC VÀ CHUẨN HÓA METADATA ---
df_all = ds_metadata_all.to_pandas().copy()
df_all["id"] = df_all["id"].astype(str)

# Chỉ giữ lại các dòng metadata có ID nằm trong file CSV
df_final = df_all[df_all["id"].isin(target_ids)].copy()

# Chuẩn hóa Metadata
df_final["legal_sectors_list"] = df_final["legal_sectors"].apply(split_sector_list)
df_final['issuance_date'] = pd.to_datetime(df_final['issuance_date'], errors='coerce')
df_final["promulgation_date"] = df_final['issuance_date'].dt.strftime('%Y-%m-%d').fillna("")
df_final["effective_date"] = "" 
df_final['year'] = df_final['issuance_date'].dt.year
df_final["signer_name"] = df_final["signers"].apply(parse_signer_name)
df_final["signer_id"] = df_final["signers"].apply(parse_signer_id)

print(f"=> TỔNG SỐ TÀI LIỆU METADATA ĐÃ LỌC: {len(df_final)}")

# --- 4. TẠO DICTIONARY CHO RAG PAYLOAD ---
metadata_by_id = {str(row['id']): row for row in df_final.to_dict(orient="records")}

# Lọc Content từ Hugging Face khớp với target_ids
print("Filtering content from Hugging Face...")
ds_content_selected = ds_content_all.filter(lambda row: str(row["id"]) in target_ids)
content_by_id = {str(row["id"]): row for row in ds_content_selected}

# --- 5. THỐNG KÊ ---
print(f"\nTotal docs requested from CSV: {len(target_ids)}")
print(f"Selected docs (Metadata) ready: {len(df_final)}")
print(f"Selected docs (Content) ready: {len(ds_content_selected)}")

Loading target IDs from CSV: /kaggle/input/datasets/nhn309261/medical-legal-docs-numbers/metadata_the_thao_y_te_100.csv
-> Found 100 IDs in CSV.

Loading dataset from Hugging Face...


/tmp/ipykernel_140/1489083882.py:51: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_final['issuance_date'] = pd.to_datetime(df_final['issuance_date'], errors='coerce')


=> TỔNG SỐ TÀI LIỆU METADATA ĐÃ LỌC: 100
Filtering content from Hugging Face...

Total docs requested from CSV: 100
Selected docs (Metadata) ready: 100
Selected docs (Content) ready: 100


# Tiếp tục từ đây: Chunk lại, lọc metadata


In [40]:
import re
import uuid
from typing import Any, Dict, List

def compact_whitespace(text: str) -> str:
    """Hàm dọn dẹp khoảng trắng dư thừa, giữ nguyên từ file cũ"""
    return re.sub(r"[ \t]+", " ", str(text or "")).strip()

class AdvancedLegalChunker:
    def __init__(self):
        # ==========================================
        # 1. BỘ NHẬN DIỆN PHÂN CẤP (HIERARCHY PATTERNS)
        # Đã nâng cấp để phục vụ Chunk Động
        # ==========================================

        # CẤP 1: Chương, Phần
        self.chapter_pattern = re.compile(r"(?im)^\s*(Chương|Phần)(?:\s+thứ)?\s+([a-zA-Z0-9]+|\d+)\b\s*[\.\:\-]?\s*(.*)$")

        # CẤP 2 (GỐC): Điều, Mục
        self.article_pattern = re.compile(r"(?im)^\s*(Điều|Mục)\s+(\d+[A-Za-z0-9\/\-]*)\s*[\.\:\-]?\s*(.*)$")

        # CẤP 3 (NHÁNH LỚN): Khoản 1, 1., 1.1., (1), hoặc gạch đầu dòng (-, +, •)
        self.clause_pattern = re.compile(
            r"(?im)^\s*(Khoản\s+\d+[\.\:\-]?)\s*(.*)$|"  # Khoản 1
            r"^\s*(\d+(?:\.\d+)*[\.\)])\s*(.*)$|"        # 1., 1.1., 1)
            r"^\s*(\(\d+\))\s*(.*)$|"                    # (1), (2)
            r"^\s*([-+•])\s+(.*)$"                       # -, +, •
        )

        # CẤP 4 (NHÁNH NHỎ): Điểm a, a)
        self.point_pattern = re.compile(r"(?im)^\s*([a-zđ]\s*[\)\.])\s*(.*)$")

        # ==========================================
        # 1.1 BỘ NHẬN DIỆN PHỤ LỤC & CẤU TRÚC TỰ DO (APPENDIX PATTERNS)
        # ==========================================
        # Cấp 1 (Lvl 1 - Article Equivalent): I., II., III. hoặc A., B., C.
        self.appx_lvl1_pattern = re.compile(r"(?im)^\s*([IVXLCDM]+|[A-Z])\s*[\.\:\-]\s*(.*)$")
        
        # Cấp 2 (Lvl 2 - Clause Equivalent): 1., 2., 3.
        self.appx_lvl2_pattern = re.compile(r"(?im)^\s*(\d+)\s*[\.\:\-]\s*(.*)$")
        
        # Cấp 3 (Lvl 3 - Point/Sub-clause Equivalent): 1.1, 1.2.1, 1.a)
        self.appx_lvl3_pattern = re.compile(r"(?im)^\s*(\d+(?:\.\d+)+)\s*[\.\:\-]?\s*(.*)$")

        # ==========================================
        # 2. BỘ NHẬN DIỆN METADATA (DỮ NGUYÊN TỪ CŨ)
        # ==========================================
        self.substantive_title_pattern = re.compile(
            r"(?im)^\s*(QUY ĐỊNH|QUY CHẾ|PHƯƠNG ÁN|ĐIỀU LỆ|CHƯƠNG TRÌNH|HƯỚNG DẪN|NỘI QUY|KẾ HOẠCH|CHIẾN LƯỢC|ĐỀ ÁN|DỰ ÁN)\b(?!\s*CHUNG\b).*$"
        )

        # Nhận diện Lời dẫn vào Phụ lục (Chặt chẽ hơn, hỗ trợ số La Mã, số Ả Rập, Ký hiệu và Các cụm từ đi kèm)
        self.appendix_title_pattern = re.compile(
            r"(?im)^\s*("
            r"(?:PHỤ\s+LỤC|PHU\s+LUC)(?:\s+(?:SỐ\s+)?(?:[IVXLCDM]+|\d+)|[A-Z])?(?:\s*[\:\-\.]|\s+BAN\s+HÀNH|\s+KÈM\s+THEO)?|"
            r"(?:MẪU|MẪU\s+SỐ|BIỂU\s+MẪU)\s*[A-Za-z0-9\.\-\/]*(?:\s*[\:\-\.]|\s+BAN\s+HÀNH|\s+KÈM\s+THEO)?|"
            r"DANH\s+MỤC(?:\s+(?:CHI\s+TIẾT|KÈM\s+THEO|CÁC|DỰ\s+ÁN|TÀI\s+SẢN|VẬT\s+TƯ|HÀNG\s+HÓA|QUỐC\s+GIA|MÃ))?"
            r")\b.*$"
        )

        self.legal_basis_line_pattern = re.compile(r"(?im)^\s*căn cứ\b.*$")
        self.legal_ref_pattern = re.compile(r"(?i)\b(Hiến pháp|Bộ luật|Luật|Nghị quyết|Pháp lệnh|Nghị định|Thông tư)\b([^.;\n]*)")

        self.effective_date_pattern = re.compile(r"(?i)có\s+hiệu\s+lực\s+(?:từ|kể\s+từ)\s+ngày\s+(\d{1,2})[/\- ](\d{1,2})[/\- ](\d{4})")
        self.effective_from_sign_pattern = re.compile(
            r"(?i)("
            r"có\s+hiệu\s+lực\s+(?:thi\s+hành\s+)?(?:kể\s+)?từ\s+ngày\s+ký|"
            r"chịu\s+trách\s+nhiệm\s+thi\s+hành\s+(?:quyết\s+định|nghị\s+định|thông\s+tư|văn\s+bản)\s+này"
            r")"
        )

        self.final_article_trigger = re.compile(
            r"(?i)("
            r"có\s+hiệu\s+lực\s+(?:thi\s+hành\s+)?(?:kể\s+)?từ\s+ngày|"
            r"chịu\s+trách\s+nhiệm\s+thi\s+hành|"
            r"tổ\s+chức\s+thực\s+hiện"
            r")"
        )

        self.footer_pattern = re.compile(
            r"(?i)^\s*(nơi\s+nhận|kính\s+gửi)[\:\.]?|"
            r"^\s*(TM\.|KT\.|Q\.|TL\.|TUQ\.)?\s*"
            r"("
            r"CHÍNH\s+PHỦ|UBND|ỦY\s+BAN\s+NHÂN\s+DÂN|"
            r"BỘ\s+TRƯỞNG|CHỦ\s+TỊCH|THỨ\s+TRƯỞNG|GIÁM\s+ĐỐC|TỔNG\s+GIÁM\s+ĐỐC|"
            r"CỤC\s+TRƯỞNG|TỔNG\s+CỤC\s+TRƯỞNG|CHÁNH\s+VĂN\s+PHÒNG|"
            r"CHÁNH\s+ÁN|VIỆN\s+TRƯỞNG|TỔNG\s+KIỂM\s+TOÁN|CHỦ\s+NHIỆM|TỔNG\s+BÍ\s+THƯ"
            r")\b"
        )

        self.relationship_pattern = re.compile(
            r"(sửa đổi(?:,\s*bổ sung)?|bổ sung|thay thế|bãi bỏ|huỷ bỏ)" # Group 1: Action
            r"(.{0,150}?)"                                              # Group 2: Scope (Lấy non-greedy, max 150 ký tự)
            r"(?:của\s+)?"                                              # Bỏ qua chữ "của" nếu có (VD: "của Luật số...")
            r"(Hiến pháp|Bộ luật|Luật|Pháp lệnh|Lệnh|Nghị quyết|Nghị định|Thông tư|Quyết định|Chỉ thị)" # Group 3: Doc Type
            r"(?:\s+số)?\s+([0-9]+/[0-9]{4}/[A-Z0-9Đ\-]+|[0-9]+/[A-Z0-9Đ\-]+)", # Group 4: Doc Number
            re.IGNORECASE
        )

        # 2. Regex bóc tách tọa độ Điều / Khoản cụ thể từ Group 2 (Scope)
        # Dùng để Neo4j vẽ Node Relationship chỉa thẳng vào đúng Khoản/Điều bị sửa đổi thay vì toàn bộ văn bản
        self.extract_article_pattern = re.compile(r"(điều\s+\d+[a-zA-ZđĐ]*)", re.IGNORECASE)
        self.extract_clause_pattern = re.compile(r"(khoản\s+\d+[a-zA-ZđĐ]*)", re.IGNORECASE)
        self.part_lesson_pattern = re.compile(r"(?im)^\s*(Phần|Tập|Bài)\s+([IVXLCDM0-9]+)\s*[\.\:\-]?\s*(.*)$")


In [41]:
# ==========================================
# CÁC HÀM TIỆN ÍCH XỬ LÝ CHUỖI (STATIC HELPERS)
# ==========================================

@staticmethod
def _slugify(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", (value or "").lower()).strip("-") or "unknown"

@staticmethod
def _canonical_doc_type(raw: str) -> str:
    text = (raw or "").lower()
    mapping = {
        "hiến pháp": "constitution", "hien phap": "constitution",
        "bộ luật": "code", "bo luat": "code",
        "luật": "law", "luat": "law",
        "pháp lệnh": "ordinance", "phap lenh": "ordinance",
        "nghị quyết": "resolution", "nghi quyet": "resolution",
        "nghị định": "decree", "nghi dinh": "decree",
        "thông tư": "circular", "thong tu": "circular"
    }
    for k, v in mapping.items():
        if k in text: return v
    return "other"

@staticmethod
def _extract_year(text: str) -> str:
    m = re.search(r"\b(19|20)\d{2}\b", text or "")
    return m.group(0) if m else ""

@staticmethod
def _extract_doc_number(text: str) -> str:
    patterns = [r"(?i)(?:số\s*)?(\d+\/\d+(?:\/[A-Z0-9Đ\-]+)?)", r"(?i)(\d+\/[A-Z0-9Đ\-]+)"]
    for p in patterns:
        m = re.search(p, text or "")
        if m: return compact_whitespace(m.group(1))
    return ""

@staticmethod
def _parse_signer(signer_raw: Any) -> tuple:
    # 1. Ép kiểu về string một cách an toàn
    signer_str = str(signer_raw) if signer_raw is not None else ""

    # 2. Nếu là giá trị rỗng hoặc NaN của Pandas
    if signer_str.strip().lower() in ["nan", "", "none"]:
        return "", None

    # 3. Xử lý phân tách như bình thường
    if ":" not in signer_str:
        return signer_str, None

    parts = signer_str.split(":")
    name = parts[0].strip()
    try:
        return name, int(parts[1].strip())
    except ValueError:
        return name, None

def normalize_doc_key(text: str) -> str:
    if not text: return ""
    # Chuyển hoa, xóa dấu cách, xóa các ký tự đặc biệt gây nhiễu
    return re.sub(r'[\s\.\-\/]', '', text).strip().upper()

# Gắn các hàm tiện ích vào Class AdvancedLegalChunker
AdvancedLegalChunker._slugify = _slugify
AdvancedLegalChunker._canonical_doc_type = _canonical_doc_type
AdvancedLegalChunker._extract_year = _extract_year
AdvancedLegalChunker._extract_doc_number = _extract_doc_number
AdvancedLegalChunker._parse_signer = _parse_signer

In [42]:
# ==========================================
# CÁC HÀM BÓC TÁCH METADATA (CĂN CỨ PHÁP LÝ & NGÀY)
# ==========================================

def _build_parent_law_id(self, doc_type: str, doc_number: str, year: str, doc_title: str) -> str:
    basis = doc_number or doc_title or "unknown"
    return f"parent::{doc_type}::{self._slugify(basis)}::{year or 'unknown'}"

def _parse_legal_basis_line(self, raw_line: str):
    refs = []
    line = compact_whitespace(raw_line)
    for m in self.legal_ref_pattern.finditer(line):
        raw_type = compact_whitespace(m.group(1))
        tail = compact_whitespace(m.group(2))
        full_ref = compact_whitespace(f"{raw_type} {tail}")
        doc_type = self._canonical_doc_type(raw_type)
        year = self._extract_year(full_ref)
        doc_number = self._extract_doc_number(full_ref)
        refs.append({
            "basis_line": line, "doc_type": doc_type, "doc_number": doc_number,
            "doc_year": year, "doc_title": full_ref,
            "parent_law_id": self._build_parent_law_id(doc_type, doc_number, year, full_ref),
        })
    return refs

def _extract_legal_basis_metadata(self, content: str) -> List[dict]:
    # Thường phần căn cứ chỉ nằm ở 80 dòng đầu tiên của văn bản
    preamble = "\n".join((content or "").splitlines()[:80])
    all_refs = []
    for line in preamble.splitlines():
        if self.legal_basis_line_pattern.match(line):
            all_refs.extend(self._parse_legal_basis_line(line))
    dedup = []
    seen = set()
    for r in all_refs:
        key = (r.get("parent_law_id"), r.get("doc_title"))
        if key in seen: continue
        seen.add(key)
        dedup.append(r)
    return dedup

def _extract_effective_date(self, content: str, promulgation_date: str) -> str:
    lines = content.splitlines()
    appendix_start_idx = len(lines)

    # Tìm điểm bắt đầu phụ lục (để không tìm nhầm ngày trong bảng biểu)
    for i, line in enumerate(lines):
        if self.appendix_title_pattern.match(line):
            appendix_start_idx = i
            break

    main_body_lines = lines[:appendix_start_idx]

    # Dùng toàn bộ nội dung chính (từ đầu đến trước Phụ lục) để quét
    search_zone = "\n".join(main_body_lines)

    m = self.effective_date_pattern.search(search_zone)
    if m:
        day, month, year = m.groups()
        return f"{year}-{month.zfill(2)}-{day.zfill(2)}"

    if self.effective_from_sign_pattern.search(search_zone):
        return promulgation_date

    return ""

# ==========================================
# YC3: HÀM BÓC TÁCH CHÍNH XÁC NỘI DUNG MỘT ĐIỀU
# ==========================================
def _extract_exact_article(self, content: str, article_name: str) -> str:
    """Bóc đúng và đủ 100% nội dung của article_name (VD: 'Điều 5') từ content.
    Lấy từ tên Điều đó kéo dài đến sát trước Điều tiếp theo hoặc Chương tiếp theo.
    Tuyệt đối không bỏ sót bất kỳ từ nào, khoản nào hay điểm nào nằm bên trong Điều đó.
    Trả về max 1500 ký tự để an toàn dung lượng."""
    if not content or not article_name:
        return ""

    # Chẻ từ và nối lại bằng \s+ để bất chấp số lượng khoảng trắng (VD: "Điều  1")
    parts = article_name.strip().split()
    safe_name = r'\s+'.join([re.escape(p) for p in parts])
    
    start_pattern = re.compile(r'(?im)^\s*' + safe_name + r'\b[\.\:\-]?\s*')
    start_match = start_pattern.search(content)
    if not start_match:
        return ""

    start_pos = start_match.start()
    remaining = content[start_match.end():]
    end_pattern = re.compile(r'(?im)^\s*(?:Điều\s+\d|Chương\s+[IVXLCDM0-9]|Phần\s+(?:thứ\s+)?[IVXLCDM0-9])')
    end_match = end_pattern.search(remaining)

    if end_match:
        result = content[start_pos: start_match.end() + end_match.start()]
    else:
        result = content[start_pos:]

    return result.strip()[:1500]

# ==========================================
# YC1+2+4: HÀM BÓC TÁCH QUAN HỆ VĂN BẢN (REFACTORED)
# ==========================================
def _extract_relationship_metadata(self, content: str, global_doc_lookup: dict = None) -> Dict[str, List[dict]]:
    lines = (content or "").splitlines()

    # ========== YC1: MÀNG LỌC PREAMBLE BẰNG ANCHOR ==========
    # Duyệt từ trên xuống, tìm dòng đầu tiên thỏa mãn:
    #   - Bắt đầu bằng "Điều 1" (VD: Điều 1., Điều 1 )
    #   - HOẶC "QUYẾT ĐỊNH:" (phải có dấu hai chấm)
    anchor_idx = None
    for i, line in enumerate(lines):
        stripped = line.strip()
        if re.match(r'(?i)^Điều\s+1\b', stripped) or re.match(r'^QUYẾT ĐỊNH\s*:', stripped):
            anchor_idx = i
            break

    # Cắt mảng lines từ Anchor trở xuống. Preamble (nửa trên) bị loại bỏ hoàn toàn.
    # Nếu không tìm thấy Anchor, giữ nguyên lines.
    search_lines = lines[anchor_idx:] if anchor_idx is not None else lines
    search_zone = "\n".join(search_lines)

    relationships = {"amended": [], "replaced": [], "repealed": []}
    seen_keys = set()

    for m in self.relationship_pattern.finditer(search_zone):
        raw_action = m.group(1).lower()
        raw_scope = m.group(2).strip()
        raw_type = m.group(3)
        doc_number = compact_whitespace(m.group(4))

        # ========== YC2: LẤY CONTEXT BLOCK ĐỘNG (new_text) ==========
        # Từ m.start() kéo dài xuống dưới cho đến khi gặp dòng bắt đầu bằng "Điều \d"
        # (Điều tiếp theo) hoặc hết chuỗi. Cắt tối đa 1500 ký tự.
        block_end_match = re.search(r'\n\s*Điều\s+\d', search_zone[m.start() + 1:])
        if block_end_match:
            new_text = search_zone[m.start(): m.start() + 1 + block_end_match.start()]
        else:
            new_text = search_zone[m.start():]
        new_text = new_text[:1500].strip()

        # QUÉT VÉT: Dùng khối text vừa lấy kết hợp raw_scope để findall Điều/Khoản
        combined_scope = raw_scope + " " + new_text

        # 1. Phân loại Action
        if "sửa" in raw_action or "bổ sung" in raw_action:
            rel_type = "amended"
        elif "thay thế" in raw_action:
            rel_type = "replaced"
        else:
            rel_type = "repealed"

        # 2. Bóc tách TỌA ĐỘ CHÍNH XÁC (Target Scope) + Deduplicate
        target_article = None
        target_clause = None

        if combined_scope:
            articles = self.extract_article_pattern.findall(combined_scope)
            if articles:
                dedup_arts = []
                for a in articles:
                    clean_a = compact_whitespace(a).capitalize()
                    if clean_a not in dedup_arts:
                        dedup_arts.append(clean_a)
                target_article = ", ".join(dedup_arts)

            clauses = self.extract_clause_pattern.findall(combined_scope)
            if clauses:
                dedup_cls = []
                for c in clauses:
                    clean_c = compact_whitespace(c).capitalize()
                    if clean_c not in dedup_cls:
                        dedup_cls.append(clean_c)
                target_clause = ", ".join(dedup_cls)

        # ========== YC2: STRICT VALIDATION ==========
        # Sửa đổi mà không chỉ đích danh Điều/Khoản nào → BẮT BUỘC loại bỏ (chống liên kết rác)
        if rel_type == "amended" and target_article is None and target_clause is None:
            continue

        # Nếu không có Điều/Khoản nào, mặc định là tác động lên Toàn bộ văn bản
        is_entire_doc = (target_article is None and target_clause is None)

        # Chống trùng lặp linh hoạt
        unique_key = f"{rel_type}_{doc_number}_{target_article}_{target_clause}"
        if unique_key in seen_keys: continue
        seen_keys.add(unique_key)

        # ========== YC4: TRUY XUẤT CHÉO old_text TỪ global_doc_lookup ==========
        old_text = ""
        doc_num_clean = doc_number.strip().upper() # CHUẨN HÓA KEY
        
        if global_doc_lookup and doc_num_clean in global_doc_lookup and target_article:
            # Lấy Điều đầu tiên trong danh sách target_article (nếu có nhiều Điều)
            first_article = target_article.split(",")[0].strip()
            if first_article:
                old_text = self._extract_exact_article(global_doc_lookup[doc_num_clean], first_article)

        relationships[rel_type].append({
            "doc_type": self._canonical_doc_type(raw_type),
            "doc_number": doc_number,
            "target_article": target_article, # Để Neo4j JOIN vào Node Article
            "target_clause": target_clause,   # Để Neo4j JOIN vào Node Chunk (Khoản)
            "is_entire_doc": is_entire_doc,
            "old_text": old_text,
            "new_text": new_text
        })

    return relationships

AdvancedLegalChunker._extract_exact_article = _extract_exact_article
AdvancedLegalChunker._extract_relationship_metadata = _extract_relationship_metadata
# Gắn các hàm bóc tách vào Class AdvancedLegalChunker
AdvancedLegalChunker._build_parent_law_id = _build_parent_law_id
AdvancedLegalChunker._parse_legal_basis_line = _parse_legal_basis_line
AdvancedLegalChunker._extract_legal_basis_metadata = _extract_legal_basis_metadata
AdvancedLegalChunker._extract_effective_date = _extract_effective_date

In [43]:
# ==========================================
# TRÁI TIM HỆ THỐNG: HÀM CẮT CHUNK (PROCESS_DOCUMENT)
# ==========================================

def process_document(self, content: str, metadata: Dict[str, Any], global_doc_lookup: dict = None) -> List[Dict[str, Any]]:
    # 1. Dọn dẹp nội dung thô
    content = str(content or "").replace("\r\n", "\n").strip()
    lines = content.splitlines()

    # 2. Khởi tạo Metadata cơ bản
    doc_id = str(metadata.get("id", uuid.uuid4()))
    doc_number = metadata.get("document_number", "N/A")
    doc_title = metadata.get("title", "N/A")
    issuing_auth = metadata.get("issuing_authority", "N/A")

    signer_name, signer_id = self._parse_signer(metadata.get("signers", ""))
    promulgation_date = metadata.get("promulgation_date", "")
    year = self._extract_year(promulgation_date) or str(metadata.get("year", "N/A"))

    eff_date = self._extract_effective_date(content, promulgation_date)
    final_effective_date = eff_date or metadata.get("effective_date", "") or promulgation_date
    basis_refs = self._extract_legal_basis_metadata(content)

    sectors_list = metadata.get("legal_sectors_list", []) # Cột mảng lĩnh vực
    sectors_str = ", ".join(sectors_list) if sectors_list else "Chung"

    # Hàm này trả về list các dict chứa: rel_type, doc_number, target_article, target_clause
    rel_dict = self._extract_relationship_metadata(content, global_doc_lookup=global_doc_lookup)
    amended_refs = rel_dict.get('amended', [])
    replaced_refs = rel_dict.get('replaced', [])
    repealed_refs = rel_dict.get('repealed', [])

    import datetime
    today_str = datetime.date.today().strftime("%Y-%m-%d")
    if final_effective_date and final_effective_date > today_str:
        doc_status = "Chưa có hiệu lực"
    else:
        doc_status = "Còn hiệu lực"

    # ==========================================
    # BƯỚC 1: TIỀN XỬ LÝ - RÚT TRÍCH MỤC LỤC (TOC) & LỜI DẪN
    # ==========================================
    toc_list = []
    found_first_structure = False

    for line_idx, line in enumerate(lines):
        line_clean = line.strip()
        if not line_clean: continue
        
        # Nếu đã chạm tới Phụ Lục, Danh Mục, Biểu Mẫu... thì KHÔNG tiếp tục đọc Mục Lục nữa để tránh lặp
        if self.appendix_title_pattern.match(line_clean):
            break

        # Quét bằng Regex để tìm Chương/Điều
        m_ch = self.chapter_pattern.match(line_clean)
        m_ar = self.article_pattern.match(line_clean)

        if m_ch:
            found_first_structure = True
            # Thêm vào TOC (VD: Chương I: Quy định chung)
            toc_list.append(f"{m_ch.group(1)} {m_ch.group(2)}: {m_ch.group(3)}")

        elif m_ar:
            found_first_structure = True
            # Thêm vào TOC (VD: Điều 1. Phạm vi điều chỉnh)
            art_num = m_ar.group(2).strip()
            art_title = m_ar.group(3).strip()
            # Giới hạn title TOC cho gọn nếu quá dài
            short_title = art_title[:100] + "..." if len(art_title) > 100 else art_title
            toc_list.append(f"  {m_ar.group(1)} {art_num}. {short_title}".strip())

    # Đóng gói TOC thành Text để chuẩn bị đưa vào Node Document trên Neo4j
    document_toc = "\n".join(toc_list)

    # Khởi tạo mảng chứa kết quả đầu ra
    chunks: List[Dict[str, Any]] = []

    # ==========================================
    # BƯỚC 2: KHỞI TẠO BIẾN TRẠNG THÁI (STATE)
    # ==========================================
    global_chunk_idx = 0
    CHUNK_LIMIT = 1200 # Giới hạn size để cắt chunk cho Khoản dài

    # ==========================================
    # BƯỚC 3: HÀM ĐÓNG GÓI CHUNK (FLUSH_ARTICLE & FLUSH_TABLE)
    # ==========================================
    def flush_article(chapter, article_ref, article_preamble, clauses_buffer, active_clause):
        nonlocal global_chunk_idx
        # Gom nốt active_clause vào buffer tạm nếu có
        _clauses = list(clauses_buffer)
        if active_clause:
            _clauses.append(active_clause)
            
        if not _clauses and not article_preamble:
            return
            
        parts = []
        if article_ref and not article_ref.isupper(): # Không nối lặp nếu là "QUY ĐỊNH CHUNG"
            parts.append(article_ref) # Bổ sung "Điều 1" hoặc "Điều 1." vào đầu text
            
        if article_preamble:
            parts.append("\n".join(article_preamble))
            
        for cl in _clauses:
            parts.append(cl["text"])
            for pt in cl.get("points", []):
                parts.append(pt)
                
        text_content = "\n".join(parts).strip()
        
        # 🟢 VÁ LỖI MIN: Lọc rác
        letters_only = re.sub(r'[\W_0-9]', '', text_content)
        # Nếu text quá ngắn (< 30 chữ cái), bỏ qua không tạo chunk
        if len(letters_only) < 30:
            return
        # Nếu chưa có Điều/Khoản mà text toàn VIẾT HOA (Tiêu đề mồ côi), bỏ qua
        if not article_ref and not clauses_buffer and text_content.isupper():
            return
        # Lọc bỏ rác mào đầu ngắn củn (Số quyết định, ngày tháng... đã có trong metadata)
        if (not article_ref or article_ref == "Lời dẫn") and not clauses_buffer and len(text_content) < 500:
            return
            
        global_chunk_idx += 1

        def group_refs(refs):
            refs = [r for r in refs if not re.match(r"^\s*[-+•]\s*$", str(r))]
            if not refs: return ""
            first = str(refs[0]).strip()
            
            m_khoan = re.match(r"(?i)(khoản)\s+(.+)", first)
            if m_khoan:
                prefix = m_khoan.group(1).capitalize()
                nums = []
                for r in refs:
                    m = re.match(r"(?i)khoản\s+(.+)", str(r).strip())
                    nums.append(m.group(1) if m else str(r).strip())
                return f"{prefix} {', '.join(nums)}"
                
            if re.match(r"^[^()]+\)$", first):
                nums = []
                for r in refs:
                    clean_r = str(r).strip()
                    m = re.match(r"^([^()]+)\)$", clean_r)
                    nums.append(m.group(1) if m else clean_r.rstrip(')'))
                return f"Điểm {', '.join(nums)})" # Tự động thêm tiền tố Điểm
                
            if re.match(r"^[^.]+\.$", first):
                nums = []
                for r in refs:
                    clean_r = str(r).strip()
                    m = re.match(r"^([^.]+)\.$", clean_r)
                    nums.append(m.group(1) if m else clean_r.rstrip('.'))
                return f"Khoản {', '.join(nums)}." # Tự động thêm tiền tố Khoản
                
            if re.match(r"^\([^()]+\)$", first):
                nums = []
                for r in refs:
                    clean_r = str(r).strip()
                    m = re.match(r"^\(([^()]+)\)$", clean_r)
                    nums.append(m.group(1) if m else clean_r.strip('()'))
                return f"Khoản ({', '.join(nums)})" # Tự động thêm tiền tố Khoản
                
            return ", ".join(str(r).strip() for r in refs)

        art_ref = article_ref if article_ref else None
        
        bc_components = [chapter, art_ref]
        clause_ref_meta = ""

        # GOM NHÓM ĐIỀU/KHOẢN
        if len(_clauses) > 1:
            cl_refs = []
            for cl in _clauses:
                ref = compact_whitespace(cl["ref"].replace(" (tiếp theo)", ""))
                if ref and ref not in cl_refs:
                    cl_refs.append(ref)
            if cl_refs:
                clause_ref_meta = group_refs(cl_refs)
                bc_components.append(clause_ref_meta)
        elif len(_clauses) == 1:
            cl_ref = compact_whitespace(_clauses[0]["ref"].replace(" (tiếp theo)", ""))
            if cl_ref:
                # Ép đi qua group_refs để lấy tiền tố Khoản/Điểm
                cl_ref_grouped = group_refs([cl_ref]) 
                clause_ref_meta = cl_ref_grouped
                bc_components.append(cl_ref_grouped)
            
            point_lines = _clauses[0].get("points", [])
            if len(point_lines) > 1:
                pt_prefs = []
                for pt in point_lines:
                    m_pt = re.match(r"^\s*([0-9]+(?:\.[0-9]+)*[\.\)]|[a-zA-ZđĐ][\.\)])", pt)
                    if m_pt:
                        prf = compact_whitespace(m_pt.group(1))
                        if prf not in pt_prefs:
                            pt_prefs.append(prf)
                if pt_prefs:
                    pt_grouped = group_refs(pt_prefs)
                    clause_ref_meta += f" - {pt_grouped}" if clause_ref_meta else pt_grouped
                    bc_components.append(pt_grouped)
            elif len(point_lines) == 1:
                m_pt = re.match(r"^\s*([0-9]+(?:\.[0-9]+)*[\.\)]|[a-zA-ZđĐ][\.\)])", point_lines[0])
                if m_pt:
                    pt_grouped = group_refs([compact_whitespace(m_pt.group(1))])
                    clause_ref_meta += f" - {pt_grouped}" if clause_ref_meta else pt_grouped
                    bc_components.append(pt_grouped)

        breadcrumb = " > ".join([x for x in bc_components if x])
        chunk_id_val = f"{doc_id}::article::c{global_chunk_idx}"
        
        short_title = doc_title[:100] + "..." if len(doc_title) > 100 else doc_title
        
        # NOTE 2: METADATA ĐẦY ĐỦ CHO LLM ĐỌC
        chunk_text = (
            f"Văn bản: {doc_number} - {short_title}\n"
            f"Lĩnh vực: {sectors_str}\n"
            f"Điều khoản: {breadcrumb}\n"
            f"Nội dung:\n"
            f"{text_content}"
        )
        
        # NOTE 2: TEXT TINH GỌN ĐỂ EMBED VÀO QDRANT (Tránh loãng ngữ nghĩa)
        text_to_embed = f"[{doc_number}] {breadcrumb}\n{text_content}"
        
        qdrant_payload = {
            "chunk_id": chunk_id_val,
            "document_id": doc_id,
            "document_number": doc_number,
            "year": year,
            "legal_sectors": sectors_list,
            "is_table": False,
            "breadcrumb": breadcrumb,
            "chunk_text": chunk_text,
            "title": doc_title
        }
        
        neo4j_payload = {
            "document_id": doc_id,
            "chunk_index": global_chunk_idx,
            "document_number": doc_number,
            "title": doc_title,
            "legal_type": metadata.get("legal_type", "N/A"),
            "legal_sectors": sectors_list,
            "issuing_authority": issuing_auth,
            "signer_name": signer_name,
            "signer_id": signer_id,
            "url": metadata.get("url", ""),
            "promulgation_date": promulgation_date,
            "effective_date": final_effective_date,
            "is_active": True,
            "chapter_ref": chapter,
            "article_ref": art_ref,
            "clause_ref": clause_ref_meta if clause_ref_meta else None,
            "is_table": False,
            "reference_citation": doc_number + (" | " + breadcrumb.replace(' > ', ' | ') if breadcrumb else ""),
            "chunk_text": chunk_text,
            "legal_basis_refs": basis_refs if global_chunk_idx == 1 else [],
            "document_toc": document_toc,
            "amended_refs": amended_refs if global_chunk_idx == 1 else [],
            "replaced_refs": replaced_refs if global_chunk_idx == 1 else [],
            "repealed_refs": repealed_refs if global_chunk_idx == 1 else [] if global_chunk_idx == 1 else []
        }
        
        chunks.append({
            "chunk_id": chunk_id_val,
            "chunk_text": chunk_text,
            "text_to_embed": text_to_embed, # Thêm trường mới
            "qdrant_metadata": qdrant_payload,
            "neo4j_metadata": neo4j_payload
        })

    def flush_table(article_ref, header_lines, row_lines):
        nonlocal global_chunk_idx
        if not row_lines:
            return
            
        global_chunk_idx += 1
        
        parts = list(header_lines) + list(row_lines)
        text_content = "\n".join(parts).strip()
        
        # FIX TABLE MIN: Nếu các hàng dữ liệu toàn ô trống (template), bỏ qua!
        letters_only = re.sub(r'[\W_0-9]', '', "\n".join(row_lines))
        if len(letters_only) < 30:
            return
        
        breadcrumb = "Dữ liệu Bảng biểu"
        if article_ref:
            breadcrumb = f"{article_ref} > {breadcrumb}"
            
        chunk_id_val = f"{doc_id}::table::c{global_chunk_idx}"
        
        short_title = doc_title[:100] + "..." if len(doc_title) > 100 else doc_title
        chunk_text = (
            f"Văn bản: {doc_number} - {short_title}\n"
            f"Lĩnh vực: {sectors_str}\n"
            f"Điều khoản: {breadcrumb}\n"
            f"Nội dung:\n"
            f"{text_content}"
        )
        text_to_embed = f"[{doc_number}] {breadcrumb}\n{text_content}"
        
        qdrant_payload = {
            "chunk_id": chunk_id_val,
            "document_id": doc_id,
            "document_number": doc_number,
            "year": year,
            "legal_sectors": sectors_list,
            "is_table": True,
            "breadcrumb": breadcrumb,
            "chunk_text": chunk_text,
            "title": doc_title
        }
        
        neo4j_payload = {
            "document_id": doc_id,
            "chunk_index": global_chunk_idx,
            "document_number": doc_number,
            "title": doc_title,
            "legal_type": metadata.get("legal_type", "N/A"),
            "legal_sectors": sectors_list,
            "issuing_authority": issuing_auth,
            "signer_name": signer_name,
            "signer_id": signer_id,
            "url": metadata.get("url", ""),
            "promulgation_date": promulgation_date,
            "effective_date": final_effective_date,
            "is_active": True,
            "chapter_ref": current_chapter,
            "article_ref": article_ref,
            "clause_ref": None,
            "is_table": True,
            "reference_citation": doc_number + (" | " + breadcrumb.replace(' > ', ' | ') if breadcrumb else ""),
            "chunk_text": chunk_text,
            "legal_basis_refs": basis_refs if global_chunk_idx == 1 else [],
            "document_toc": document_toc,
            "amended_refs": amended_refs if global_chunk_idx == 1 else [],
            "replaced_refs": replaced_refs if global_chunk_idx == 1 else [],
            "repealed_refs": repealed_refs if global_chunk_idx == 1 else [] if global_chunk_idx == 1 else []
        }
        
        chunks.append({
            "chunk_id": chunk_id_val,
            "chunk_text": chunk_text,
            "qdrant_metadata": qdrant_payload,
            "neo4j_metadata": neo4j_payload,
            "text_to_embed": text_to_embed
        })

    # --- PHẦN 4: VÒNG LẶP DUYỆT TEXT CHÍNH ---

    def compact_whitespace(text: str) -> str:
        if not text: return ""
        text = str(text).replace('\r', '')
        return ' '.join(text.split())

    def get_clause_size(clause_obj) -> int:
        if not clause_obj: return 0
        return len(clause_obj["text"]) + sum(len(p) for p in clause_obj["points"])

    lines = content.splitlines()

    # 1. KHỞI TẠO CÁC "KHAY CHỨA" (BUFFERS)
    in_table = False
    table_header = []
    table_rows = []

    current_chapter = None
    current_article_ref = None         # Lời dẫn của Điều (Context Injection)
    current_article_preamble = []      # Bộ nhớ 1
    current_clauses_buffer = []        # Lưu các Khoản đã hoàn thiện
    current_active_clause = None       # Bộ nhớ 2 + Danh sách Điểm đang thu thập
    
    in_appendix = False                # Cờ trạng thái: Đang ở trong khu vực Phụ Lục

    for line_idx, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue

        # ==============================================================
        # BƯỚC 2: BẢO VỆ VÙNG AN TOÀN (Header Protection & Table Processing)
        # ==============================================================
        is_safe_zone = (line_idx < 50)
        if is_safe_zone and (line.isupper() and "ĐỘC LẬP" in line or "TỰ DO" in line):
            continue
            
        # Lọc bỏ hẳn các dòng Footer không cần thiết (Ký tên, Nơi nhận...)
        if self.footer_pattern.match(line):
            continue

        if line.count('|') >= 2:
            if not in_table:
                flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
                current_clauses_buffer = []
                current_active_clause = None

                in_table = True
                table_header = [line]
                table_rows = []
                continue
            else:
                if set(line.replace('|', '').replace('-', '').replace(':', '').strip()) == set():
                    table_header.append(line)
                    continue

                if len(table_header) < 2 and not table_rows:
                    table_header.append(line)
                    continue

                if table_rows:
                    current_table_len = sum(len(r) for r in table_header) + sum(len(r) for r in table_rows)
                    # Nếu thêm dòng này vào mà làm bảng lố 3000 ký tự -> Bắn cục cũ đi đã!
                    if len(table_rows) >= 20 or (current_table_len + len(line)) > 3000:
                        flush_table(current_article_ref, table_header, table_rows)
                        table_rows = [] 
                
                table_rows.append(line)
                
                # CHỐNG TRÀN DÒNG NGOẠI CỠ: Nếu tự thân 1 dòng bảng đã > 3000 ký tự, bắn đi luôn để cách ly
                if len(table_rows) == 1 and len(line) > 3000:
                    flush_table(current_article_ref, table_header, table_rows)
                    table_rows = [] 
            continue
        else:
            if in_table:
                if table_rows:
                    flush_table(current_article_ref, table_header, table_rows)
                in_table = False
                table_header, table_rows = [], []

        # ==============================================================
        # BƯỚC 3: NHẬN DIỆN CẤU TRÚC PHÂN CẤP LUẬT CHUẨN (VỊ TRÍ 1 - MÀNG LỌC VUA)
        # Ưu tiên nhận diện Chương -> Điều -> Khoản -> Điểm bám sát bất kỳ đâu
        # ==============================================================
        
        # 3.1. NHẬN DIỆN CHƯƠNG
        m_ch = self.chapter_pattern.match(line)
        if m_ch:
            flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
            current_chapter = compact_whitespace(f"{m_ch.group(1)} {m_ch.group(2)}")
            current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause = None, [], [], None
            in_appendix = False # Thoát phụ lục nếu gặp Chương mới của Luật
            continue

        # 3.2. NHẬN DIỆN ĐIỀU
        m_ar = self.article_pattern.match(line)
        if m_ar:
            # 🟢 VÁ LỖI MIN: Nếu buffer cũ chỉ là Tiêu đề Chương ngắn (<150 ký tự), KHÔNG FLUSH, gộp luôn vào Điều này!
            buffer_len = sum(len(p) for p in current_article_preamble) + sum(get_clause_size(c) for c in current_clauses_buffer)
            if not current_clauses_buffer and not current_active_clause and buffer_len < 150:
                pass 
            else:
                flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
                current_article_preamble = []

            # Sửa lỗi: group(1) chỉ là "Điều", cần ghép thêm group(2) để có "Điều 1"
            current_article_ref = f"{m_ar.group(1).strip()} {m_ar.group(2).strip()}"
            article_title = m_ar.group(3).strip()
            
            if article_title:
                current_article_preamble.append(article_title)
            current_clauses_buffer = []
            current_active_clause = None
            in_appendix = False # Thoát phụ lục nếu có Điều mới của Luật
            continue
            
        # 3.3. NHẬN DIỆN KHOẢN (VÍ DỤ: "1. Lời dẫn khoản...")
        m_cl = self.clause_pattern.match(line)
        if m_cl and current_article_ref and not in_appendix:
            if current_active_clause:
                active_size = get_clause_size(current_active_clause)
                buffer_size = sum(get_clause_size(c) for c in current_clauses_buffer)
                
                # Nếu vụt mốc CHUNK_LIMIT
                if (buffer_size + active_size) > CHUNK_LIMIT:
                    if buffer_size > 0: # Flush cái giỏ cũ đi, giữ lại Khoản hiện tại cho giỏ mới
                        flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, None)
                        current_clauses_buffer = [current_active_clause]
                    else: # Giỏ rỗng mà bản thân Khoản quá bự -> Bắn đi luôn
                        flush_article(current_chapter, current_article_ref, current_article_preamble, [], current_active_clause)
                        current_clauses_buffer = []
                    
                    if current_article_preamble:
                        current_article_preamble = [] # Dọn rác
                else:
                    current_clauses_buffer.append(current_active_clause)

            cl_ref = compact_whitespace(m_cl.group(1) or m_cl.group(3) or m_cl.group(5) or m_cl.group(7))
            cl_text = compact_whitespace(m_cl.group(2) or m_cl.group(4) or m_cl.group(6) or m_cl.group(8))

            # BỘ NHỚ 2: Lời dẫn của Khoản (Bao gồm ref và text gốc của Khoản chưa có điểm)
            current_active_clause = {
                "ref": cl_ref,
                "text": f"{cl_ref} {cl_text}".strip(),
                "points": [] 
            }
            continue

        # 3.4. NHẬN DIỆN ĐIỂM (VÍ DỤ: "a) Cơ quan nhà nước...")
        m_pt = self.point_pattern.match(line)
        if m_pt and current_active_clause and not in_appendix:
            pt_ref = compact_whitespace(m_pt.group(1))
            pt_text = compact_whitespace(m_pt.group(2))
            new_point_text = f"{pt_ref} {pt_text}"
            
            buffer_sz = sum(get_clause_size(c) for c in current_clauses_buffer)
            # TRIGGER LIMIT: Cắt chunk ngay nếu việc thêm Điểm này, kể cả khi Clause Preamble chưa có Điểm nào
            if buffer_sz + get_clause_size(current_active_clause) + len(new_point_text) > CHUNK_LIMIT:
                # 1. Kích hoạt Lệnh Cắt (Flush toàn bộ Điều và Khoản hiện tại với các Điểm nó đang chứa)
                flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
                # 2. Xóa sạch Buffer
                current_clauses_buffer = []
                if current_article_preamble:
                    current_article_preamble = []
                
                # 3. TIÊM BỘ NHỚ 2 (Inject): Lấy lời dẫn của khoản trước đó thêm chữ "(tiếp theo)"
                cl_ref_base = current_active_clause["ref"].replace(" (tiếp theo)", "")
                
                current_active_clause = {
                    "ref": f"{cl_ref_base} (tiếp theo)",
                    "text": f"[{cl_ref_base} tiếp theo]",
                    "points": [new_point_text] # Bắt đầu cục chunk mới bằng Điểm vừa bị kẹt
                }
            else:
                # Nếu chưa vượt Limit, tiếp tục gom Điểm vào Chunk
                current_active_clause["points"].append(new_point_text)
            continue

        # ==============================================================
        # BƯỚC 4: NHẬN DIỆN PHỤ LỤC & HƯỚNG DẪN CHUYÊN MÔN (KÈM THEO)
        # ==============================================================
        
        # 1. Lối vào: Có thể là "PHỤ LỤC" hoặc "HƯỚNG DẪN / QUY ĐỊNH..."
        m_appx_title = self.appendix_title_pattern.match(line) or self.substantive_title_pattern.match(line)
        # CHỐNG NHẬN DIỆN LỖI: Tiêu đề Hướng dẫn/Quy định không bao giờ dài hàng ngàn ký tự. Nếu > 200 => Đây là đoạn văn.
        if m_appx_title and len(line) < 200:
            flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
            in_appendix = True
            # Lấy nguyên dòng tiêu đề (VD: "HƯỚNG DẪN CHẨN ĐOÁN...") làm Chapter
            current_chapter = compact_whitespace(line) 
            current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause = None, [], [], None
            continue

        # NẾU ĐANG TRONG LÃNH THỔ HƯỚNG DẪN / PHỤ LỤC
        if in_appendix:
            # 4.A1 Nhận diện Tập / Phần / Bài (Tương đương Điều)
            m_part = self.part_lesson_pattern.match(line)
            if m_part:
                flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
                current_article_ref = f"{m_part.group(1)} {m_part.group(2)}"
                title_part = m_part.group(3).strip()
                current_article_preamble = [title_part] if title_part else []
                current_clauses_buffer, current_active_clause = [], None
                continue
                
            # 4.A2 Nhận diện Tiêu đề Khối (Viết hoa toàn bộ, VD: "LỜI GIỚI THIỆU", "MỤC LỤC")
            # Điều kiện: Chữ hoa toàn bộ, dài từ 5-150 ký tự, không bắt đầu bằng số (để tránh nhầm với I, II, III)
            if line.isupper() and 5 < len(line) < 150 and not re.match(r'^[IVXLCDM0-9]', line):
                flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
                current_article_ref = line.strip() # Dùng "LỜI GIỚI THIỆU" làm tọa độ Điều luôn
                current_article_preamble = []
                current_clauses_buffer, current_active_clause = [], None
                continue

            # 4.A3 CẤP 1 PHỤ LỤC: I, II, III hoặc A, B, C
            m_a1 = self.appx_lvl1_pattern.match(line)
            if m_a1:
                flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
                current_article_ref = compact_whitespace(f"{m_a1.group(1)} {m_a1.group(2)}")
                current_article_preamble = []
                current_clauses_buffer, current_active_clause = [], None
                continue

            # 4.B CẤP 2 PHỤ LỤC: 1, 2, 3
            m_a2 = self.appx_lvl2_pattern.match(line)
            if m_a2:
                if current_active_clause:
                    active_size = get_clause_size(current_active_clause)
                    buffer_size = sum(get_clause_size(c) for c in current_clauses_buffer)

                    if (buffer_size + active_size) > CHUNK_LIMIT:
                        if buffer_size > 0:
                            flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, None)
                            current_clauses_buffer = [current_active_clause]
                        else:
                            flush_article(current_chapter, current_article_ref, current_article_preamble, [], current_active_clause)
                            current_clauses_buffer = []
                            
                        if current_article_preamble:
                            current_article_preamble = []
                    else:
                        current_clauses_buffer.append(current_active_clause)

                cl_ref = compact_whitespace(m_a2.group(1))
                cl_text = compact_whitespace(m_a2.group(2))
                
                # Sử dụng cấp Clause làm Bộ nhớ 2 (Level 2)
                current_active_clause = {
                    "ref": f"{cl_ref}.",
                    "text": f"{cl_ref}. {cl_text}".strip(),
                    "points": [] 
                }
                continue

            # 4.C CẤP 3 PHỤ LỤC (Sub-clause / Point): 1.1, 1.2.1 hoặc a), b)
            m_a3 = self.appx_lvl3_pattern.match(line)
            m_pt_appx = self.point_pattern.match(line) # Dùng lại pattern cũ cho Phụ Lục
            
            sub_match = m_a3 or m_pt_appx
            if sub_match and current_active_clause:
                pt_ref = compact_whitespace(sub_match.group(1))
                pt_text = compact_whitespace(sub_match.group(2))
                new_point_text = f"{pt_ref} {pt_text}"
                
                buffer_sz = sum(get_clause_size(c) for c in current_clauses_buffer)
                # TRIGGER LIMIT cho Level 3
                if buffer_sz + get_clause_size(current_active_clause) + len(new_point_text) > CHUNK_LIMIT:
                    flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
                    current_clauses_buffer = []
                    if current_article_preamble:
                        current_article_preamble = []
                    cl_ref_base = current_active_clause["ref"].replace(" (tiếp theo)", "")
                    current_active_clause = {
                        "ref": f"{cl_ref_base} (tiếp theo)",
                        "text": f"[{cl_ref_base} tiếp theo]",
                        "points": [new_point_text]
                    }
                else:
                    current_active_clause["points"].append(new_point_text)
                continue
                
        # ==============================================================
        # BƯỚC 5: TEXT TỰ DO (ĐOẠN VĂN TIẾP NỐI) - VỊ TRÍ MÀNG LỌC CUỐI
        # ==============================================================
        TEXT_LIMIT = 1500
        
        # 🟢 VÁ LỖI LỌC RÁC MÀO ĐẦU: Xóa các dòng "Căn cứ..." ở phần mở đầu chưa vào nội dung
        if not current_chapter and not current_article_ref and line.lower().startswith("căn cứ"):
            continue

        if current_active_clause:
            # Preamble size check to avoid bloating
            preamble_size = sum(len(p) for p in current_article_preamble) if current_article_preamble else 0
            buffer_size = sum(get_clause_size(c) for c in current_clauses_buffer)
            
            # KIỂM TRA TRƯỚC KHI NẠP: Nếu Clause (và lượng preamble + buffer ký sinh) vượt quá TEXT_LIMIT
            if buffer_size + preamble_size + get_clause_size(current_active_clause) + len(line) > TEXT_LIMIT:
                flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
                current_clauses_buffer = []
                # Đóng băng Preamble để khỏi dính lây mầm bệnh vào mọi chunk kế tiếp
                if current_article_preamble:
                    current_article_preamble = []
                    
                cl_ref_base = current_active_clause["ref"].replace(" (tiếp theo)", "")
                
                # Tạo Chunk nối tiếp
                if current_active_clause["points"]:
                    current_active_clause = {
                        "ref": f"{cl_ref_base} (tiếp theo)",
                        "text": f"[{cl_ref_base} tiếp theo]",
                        "points": [line]
                    }
                else:
                    current_active_clause = {
                        "ref": f"{cl_ref_base} (tiếp theo)",
                        "text": f"[{cl_ref_base} tiếp theo]\n{line}",
                        "points": []
                    }
            else:
                if current_active_clause["points"]:
                    current_active_clause["points"][-1] += f"\n{line}"
                else:
                    current_active_clause["text"] += f"\n{line}"
        elif current_article_ref:
            if sum(len(p) for p in current_article_preamble) + len(line) > TEXT_LIMIT:
                flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
                current_article_preamble = [line]
                current_clauses_buffer = []
                current_active_clause = None
            else:
                current_article_preamble.append(line)
        elif current_chapter:
             # Nếu đang có Chương/Phụ lục mà chưa có Điều/Cấp 1
            if sum(len(p) for p in current_article_preamble) + len(line) > TEXT_LIMIT:
                flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
                current_article_preamble = [line]
                current_clauses_buffer = []
                current_active_clause = None
            else:
                current_article_preamble.append(line)
        else:
            # GOM TEXT TỰ DO NẾU NÓ KHÔNG CÓ BẤT KỲ CẤU TRÚC NÀO (PHẦN MỞ ĐẦU)
            if sum(len(p) for p in current_article_preamble) + len(line) > TEXT_LIMIT:
                flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)
                current_article_preamble = [line]
                current_clauses_buffer = []
                current_active_clause = None
            else:
                current_article_preamble.append(line)

    # --- KẾT THÚC VÒNG LẶP: FLUSH NỐT DỮ LIỆU CUỐI ---
    if in_table:
        flush_table(current_article_ref, table_header, table_rows)
    else:
        flush_article(current_chapter, current_article_ref, current_article_preamble, current_clauses_buffer, current_active_clause)

    return chunks

# Gắn hàm xử lý lõi vào Class
AdvancedLegalChunker.process_document = process_document

In [44]:
# Khởi tạo đối tượng Chunker
chunker = AdvancedLegalChunker()
print("✅ Advanced Chunker with Context Injection & Fine-grained points is ready (Modularized).")

✅ Advanced Chunker with Context Injection & Fine-grained points is ready (Modularized).


In [45]:
import time

def estimate_total_chunks(dataset, metadata_dict):
    print("🚀 Đang quét toàn bộ dữ liệu để tính toán quy mô... (Chỉ chạy CPU, không tốn GPU)")
    t_start = time.perf_counter()

    total_docs = len(dataset)
    total_chunks = 0
    doc_count = 0

    # --- Kỷ lục cho NỘI DUNG CHÍNH (is_table = False) ---
    main_max = {"len": 0, "doc": "", "pos": "", "text": ""}
    main_min = {"len": float('inf'), "doc": "", "pos": "", "text": ""}

    # --- Kỷ lục cho BẢNG BIỂU (is_table = True) ---
    table_max = {"len": 0, "doc": "", "pos": "", "text": ""}
    table_min = {"len": float('inf'), "doc": "", "pos": "", "text": ""}

    # Chỉ chạy logic cắt text, không nhúng
    for row in dataset:
        doc_id = str(row["id"])
        meta = metadata_dict.get(doc_id)
        if not meta:
            continue

        text_content = str(row.get("text", row.get("content", "")))

        # Gọi hàm chunker
        doc_chunks = chunker.process_document(text_content, meta)
        total_chunks += len(doc_chunks)
        doc_count += 1

        # --- KIỂM TRA TỪNG CHUNK ---
        for chunk in doc_chunks:
            current_chunk_text = chunk.get("chunk_text", "")
            current_len = len(current_chunk_text)
            chunk_meta = chunk.get("neo4j_metadata", chunk.get("metadata", {}))

            is_table = chunk_meta.get("is_table", False)
            doc_num = chunk_meta.get("document_number", "N/A")
            ref_cit = chunk_meta.get("reference_citation", "N/A")

            # Phân luồng cập nhật kỷ lục
            if is_table:
                # Kỷ lục Bảng biểu
                if current_len > table_max["len"]:
                    table_max = {"len": current_len, "doc": doc_num, "pos": ref_cit, "text": current_chunk_text}
                if current_len < table_min["len"]:
                    table_min = {"len": current_len, "doc": doc_num, "pos": ref_cit, "text": current_chunk_text}
            else:
                # Kỷ lục Nội dung chính
                if current_len > main_max["len"]:
                    main_max = {"len": current_len, "doc": doc_num, "pos": ref_cit, "text": current_chunk_text}
                if current_len < main_min["len"]:
                    main_min = {"len": current_len, "doc": doc_num, "pos": ref_cit, "text": current_chunk_text}

        if doc_count % 500 == 0:
            print(f"--- Đã quét {doc_count}/{total_docs} văn bản... ---")

    duration = time.perf_counter() - t_start

    # Xử lý trường hợp mảng rỗng (Inf)
    if main_min["len"] == float('inf'): main_min["len"] = 0
    if table_min["len"] == float('inf'): table_min["len"] = 0

    # Hàm in mẫu report để code gọn gàng
    def print_record(title, record, icon):
        print("\n" + "="*60)
        print(f"{icon} {title}")
        print("="*60)
        if record['len'] == 0:
            print("Không có dữ liệu trong nhóm này.")
        else:
            print(f"🔸 Số ký tự: {record['len']:,} chars")
            print(f"🔸 Văn bản: {record['doc']}")
            print(f"🔸 Vị trí: {record['pos']}")
            print("-" * 60)
            print("🔸 NỘI DUNG CHUNK:\n")
            print(record['text'])

    # --- IN BÁO CÁO TỔNG QUAN ---
    print("\n" + "="*60)
    print("📊 BÁO CÁO DỰ KIẾN (ESTIMATION REPORT)")
    print("="*60)
    print(f"🔹 Tổng số văn bản: {doc_count:,}")
    print(f"🔹 Tổng số chunk dự kiến: {total_chunks:,}")
    if doc_count > 0:
        print(f"🔹 Tỷ lệ trung bình: {total_chunks/doc_count:.2f} chunks/văn bản")
    print(f"⏱️ Thời gian quét: {duration:.2f} giây")
    est_size_mb = (total_chunks * 10) / 1024
    print(f"💾 Ước tính dung lượng DB: ~{est_size_mb:.2f} MB")

    # --- IN CHI TIẾT KỶ LỤC ---
    print_record("NỘI DUNG CHÍNH: CHUNK DÀI NHẤT (MAX)", main_max, "⚠️")
    print_record("NỘI DUNG CHÍNH: CHUNK NGẮN NHẤT (MIN)", main_min, "🔎")

    print_record("BẢNG BIỂU: CHUNK DÀI NHẤT (MAX)", table_max, "⚠️")
    print_record("BẢNG BIỂU: CHUNK NGẮN NHẤT (MIN)", table_min, "🔎")

    print("="*60)

# Chạy thử
estimate_total_chunks(ds_content_selected, metadata_by_id)

🚀 Đang quét toàn bộ dữ liệu để tính toán quy mô... (Chỉ chạy CPU, không tốn GPU)

📊 BÁO CÁO DỰ KIẾN (ESTIMATION REPORT)
🔹 Tổng số văn bản: 100
🔹 Tổng số chunk dự kiến: 3,352
🔹 Tỷ lệ trung bình: 33.52 chunks/văn bản
⏱️ Thời gian quét: 1.36 giây
💾 Ước tính dung lượng DB: ~32.73 MB

⚠️ NỘI DUNG CHÍNH: CHUNK DÀI NHẤT (MAX)
🔸 Số ký tự: 2,667 chars
🔸 Văn bản: 55/2025/TT-BYT
🔸 Vị trí: 55/2025/TT-BYT | Chương IV | Điều 8 | Khoản 6, 7.
------------------------------------------------------------
🔸 NỘI DUNG CHUNK:

Văn bản: 55/2025/TT-BYT - Thông tư 55/2025/TT-BYT quy định về kê đơn thuốc cổ truyền, thuốc dược liệu và kê đơn kết hợp thuốc ...
Lĩnh vực: Thể thao - Y tế
Điều khoản: Chương IV > Điều 8 > Khoản 6, 7.
Nội dung:
Điều 8
NỘI DUNG ĐƠN THUỐC, SỬ DỤNG THUỐC VÀ THỜI HẠN CỦA ĐƠN THUỐC
Yêu cầu chung với nội dung kê đơn thuốc, hồ sơ bệnh án
Ngày 01 thang x 07 ngày.
Sắc uống ngày 01 thang, chia 02 lần sáng và chiều, uống sau ăn.
b) Người hành nghề kê đơn thuốc thang riêng lẻ hoặc kê đơn kết hợp 

In [46]:
# @title Cell Test Parser: Kiểm tra văn bản theo SỐ HIỆU (document_number)
import json

def run_test_by_doc_number(target_number="48/2026/NĐ-CP"):
    try:
        # 1. Kiểm tra dữ liệu đã load chưa
        if 'content_by_id' not in globals() or 'metadata_by_id' not in globals():
            print("❌ Lỗi: Chưa load dữ liệu từ Hugging Face. Hãy chạy cell load dataset trước!")
            return

        print(f"🔍 Đang tìm kiếm văn bản có số hiệu: {target_number}...")

        # Tìm ID tương ứng với document_number
        target_doc_id = None
        for doc_id, meta in metadata_by_id.items():
            if meta.get('document_number') == target_number:
                target_doc_id = doc_id
                break

        if not target_doc_id:
            print(f"⚠️ Không tìm thấy văn bản có số hiệu {target_number} trong danh sách đã load.")
            print("Gợi ý: Kiểm tra lại dấu cách hoặc chính tả (VD: 142/QĐ-QLD thay vì 142/QD-QLD)")
            return

        target_meta = metadata_by_id[target_doc_id]
        content_dict = content_by_id[target_doc_id]
        target_content = content_dict.get('text', content_dict.get('content', ''))

        print(f"✅ Đã tìm thấy văn bản ID: {target_doc_id}")
        print(f"📊 Tiêu đề: {target_meta.get('title')}")
        print("-" * 80)

        # 2. Chạy Chunker (Đảm bảo biến chunker đã được khởi tạo từ class AdvancedLegalChunker mới)
        test_chunks = chunker.process_document(target_content, target_meta)
        print(f"🚀 Tổng số chunk cắt được: {len(test_chunks)}")

        if len(test_chunks) == 0:
            print("⚠️ Văn bản này không cắt được chunk nào.")
            return

        # 3. In toàn bộ Chunk
        print(f"\n🔍 SOI CHI TIẾT TOÀN BỘ {len(test_chunks)} CHUNK:")

        for i, c in enumerate(test_chunks):
            meta = c.get('neo4j_metadata', c.get('metadata', {}))
            print("=" * 80)
            print(f"📦 CHUNK INDEX: {meta.get('chunk_index')} | ID: {c.get('chunk_id')}")
            print(f"🔗 TRÍCH DẪN (Breadcrumb): {meta.get('reference_citation')}")
            print(f"📎 KHOẢN (clause_ref): {meta.get('clause_ref')}")
            sectors = meta.get('legal_sectors')
            print(f"📎 LĨNH VỰC: {', '.join(sectors) if sectors else 'Chung'}")
            print(f"📎 CỜ BẢNG BIỂU (is_table): {meta.get('is_table')}")
            print("-" * 80)
            print(f"📝 TEXT ĐỀ ĐƯA VÀO MODEL NHÚNG:\n{c.get('chunk_text')}")
        
    except Exception as e:
        print(f"❌ Có lỗi xảy ra trong quá trình chạy test: {str(e)}")

# Chạy test với một văn bản nào đó (ví dụ Nghị định)
# run_test_by_doc_number("80/2024/NĐ-CP")
run_test_by_doc_number("142/2024/NĐ-CP")

🔍 Đang tìm kiếm văn bản có số hiệu: 142/2024/NĐ-CP...
⚠️ Không tìm thấy văn bản có số hiệu 142/2024/NĐ-CP trong danh sách đã load.
Gợi ý: Kiểm tra lại dấu cách hoặc chính tả (VD: 142/QĐ-QLD thay vì 142/QD-QLD)


In [47]:
import time

def analyze_table_ratio(dataset, metadata_dict):
    print("🔍 Đang quét dữ liệu để phân tích tỷ lệ Bảng biểu / Nội dung chính...")
    t_start = time.perf_counter()

    total_docs = len(dataset)
    total_chunks = 0
    table_chunks = 0
    main_chunks = 0
    doc_count = 0

    for row in dataset:
        doc_id = str(row["id"])
        meta = metadata_dict.get(doc_id)
        if not meta:
            continue

        text_content = str(row.get("text", row.get("content", "")))

        # Gọi chunker để cắt thử
        doc_chunks = chunker.process_document(text_content, meta)

        for chunk in doc_chunks:
            total_chunks += 1
            chunk_meta = chunk.get("neo4j_metadata", chunk.get("metadata", {}))
            
            # Kiểm tra cờ is_table
            if chunk_meta.get("is_table") == True:
                table_chunks += 1
            else:
                main_chunks += 1

        doc_count += 1
        if doc_count % 500 == 0:
            print(f"--- Đã quét {doc_count}/{total_docs} văn bản... ---")

    duration = time.perf_counter() - t_start

    # Tính phần trăm
    table_percent = (table_chunks / total_chunks) * 100 if total_chunks > 0 else 0
    main_percent = (main_chunks / total_chunks) * 100 if total_chunks > 0 else 0

    print("\n" + "="*60)
    print("📊 BÁO CÁO TỶ LỆ BẢNG BIỂU (TABLE RATIO REPORT)")
    print("="*60)
    print(f"🔹 Tổng số văn bản đã quét: {doc_count:,}")
    print(f"🔹 Tổng số chunks sinh ra:  {total_chunks:,}")
    print("-" * 60)
    print(f"🟢 NỘI DUNG CHÍNH (is_table=False): {main_chunks:>7,} chunks ({main_percent:>5.2f}%)")
    print(f"🟡 BẢNG BIỂU (is_table=True):       {table_chunks:>7,} chunks ({table_percent:>5.2f}%)")
    print("-" * 60)
    print(f"⏱️ Thời gian tính toán: {duration:.2f} giây")
    print("="*60)

    # Đưa ra chẩn đoán dựa trên tỷ lệ
    if table_percent > 35:
        print("⚠️ CẢNH BÁO ĐỎ: DB của bạn đang bị 'ngập' bảng biểu!")
        print("💡 Nguy cơ: Tốn dung lượng Qdrant, làm loãng kết quả tìm kiếm ngữ nghĩa.")
    elif table_percent > 15:
        print("⚡ LƯU Ý: Tỷ lệ bảng biểu ở mức trung bình cao. Cần dùng Filter khi truy vấn.")
    else:
        print("✅ TUYỆT VỜI: Tỷ lệ nội dung chính áp đảo, DB rất 'sạch' và chất lượng!")

# Chạy thử nghiệm
analyze_table_ratio(ds_content_selected, metadata_by_id)

🔍 Đang quét dữ liệu để phân tích tỷ lệ Bảng biểu / Nội dung chính...

📊 BÁO CÁO TỶ LỆ BẢNG BIỂU (TABLE RATIO REPORT)
🔹 Tổng số văn bản đã quét: 100
🔹 Tổng số chunks sinh ra:  3,352
------------------------------------------------------------
🟢 NỘI DUNG CHÍNH (is_table=False):   1,998 chunks (59.61%)
🟡 BẢNG BIỂU (is_table=True):         1,354 chunks (40.39%)
------------------------------------------------------------
⏱️ Thời gian tính toán: 1.27 giây
⚠️ CẢNH BÁO ĐỎ: DB của bạn đang bị 'ngập' bảng biểu!
💡 Nguy cơ: Tốn dung lượng Qdrant, làm loãng kết quả tìm kiếm ngữ nghĩa.


In [51]:
from neo4j import GraphDatabase

def enrich_reference_nodes(driver, batch_chunks, meta_by_docnum_lookup):
    missing_docs = set()
    for chunk in batch_chunks:
        meta = chunk.get("neo4j_metadata", {})
        # Gom sạch từ mọi nguồn quan hệ
        all_refs = (meta.get("legal_basis_refs", []) + 
                    meta.get("amended_refs", []) + 
                    meta.get("replaced_refs", []) + 
                    meta.get("repealed_refs", []))
        for ref in all_refs:
            dnum = ref.get("doc_number")
            if dnum and dnum not in ["unknown", "N/A"]:
                missing_docs.add(dnum.strip())

    enrich_payloads = []
    for dnum in missing_docs:
        key = normalize_doc_key(dnum) # Chuẩn hóa để tra cứu
        if key in meta_by_docnum_lookup:
            hf = meta_by_docnum_lookup[key]
            
            # Format ngày tháng an toàn
            p_date = str(hf.get("issuance_date", "")).split("T")[0] if hf.get("issuance_date") else ""
            
            enrich_payloads.append({
                "document_number": dnum, # Giữ nguyên số hiệu gốc để MERGE
                "id": str(hf.get("id", f"REF_{dnum}")),
                "title": hf.get("title", ""),
                "url": hf.get("url", ""),
                "p_date": p_date,
                "eff_date": str(hf.get("effective_date", p_date)), # Fallback về ngày ban hành
                "doc_toc": hf.get("document_toc", ""),
                "year": p_date[:4] if p_date else str(hf.get("year", "")),
                "doc_status": "Còn hiệu lực" # Mặc định, Cypher sẽ cập nhật nếu bị bãi bỏ
            })

    if not enrich_payloads: return

    query = """
    UNWIND $batch AS doc
    MERGE (p:Document {document_number: doc.document_number})
    // Chỉ cập nhật nếu nốt đó đang thiếu dữ liệu (nốt ảo)
    WITH p, doc
    WHERE p.title IS NULL OR p.title = '' OR p.id STARTS WITH 'REF_'
    SET p.id = doc.id,
        p.title = doc.title,
        p.url = doc.url,
        p.promulgation_date = doc.p_date,
        p.effective_date = doc.eff_date,
        p.document_toc = doc.doc_toc,
        p.year = doc.year,
        p.doc_status = COALESCE(p.doc_status, doc.doc_status),
        p.is_full_text = false
    """
    with driver.session() as session:
        session.run(query, batch=enrich_payloads)

def build_neo4j(driver, batch_chunks):
    """Push batch of chunks into Neo4j using UNWIND for massive speed up với Đồ Thị Động"""
    if not batch_chunks: return
    # 🟢 CHẠY ENRICHMENT TRƯỚC TIÊN\n
    enrich_reference_nodes(driver, batch_chunks, meta_by_docnum_lookup)

    # Bộ nhớ đánh dấu các Điều/Khoản đã nạp text, dùng để tránh ghi đè Đồ thị
    seen_nodes = set()

    # 1. ĐÓNG GÓI DATA TẠI PYTHON: Xác định Vai Trò (Leaf Level)
    params = []
    for chunk in batch_chunks:
        meta = chunk.get("neo4j_metadata", chunk.get("metadata", {}))
        p_date = meta.get("promulgation_date", "")
        year = p_date[:4] if p_date and len(p_date) >= 4 else ""
        
        art_ref = meta.get("article_ref")
        cl_ref_raw = meta.get("clause_ref")
        is_table = meta.get("is_table", False)
        
        # Logic Đồ thị Động: Phân loại Mức độ Lá (Leaf Level)
        is_article_leaf = False
        is_clause_leaf = False
        is_chunk_leaf = False
        base_clause_ref = None

        if art_ref:
            if not cl_ref_raw:
                node_key = f"ART_{meta.get('document_id')}_{art_ref}"
                if is_table or node_key in seen_nodes:
                    # Nếu là bảng biểu hoặc Nút Điều đã có text -> Ép thành Chunk con
                    is_chunk_leaf = True
                else:
                    # Kịch bản A: Điều là lá (Không có khoản)
                    is_article_leaf = True
                    seen_nodes.add(node_key)
            else:
                if "(tiếp theo)" in cl_ref_raw or "[" in cl_ref_raw:
                    # Kịch bản C: Chunk là lá (Bị cắt vỡ từ một Khoản, hoặc bị ngắt đoạn Lời dẫn)
                    is_chunk_leaf = True
                    base_clause_ref = cl_ref_raw.replace(" (tiếp theo)", "").replace("[", "").replace(" tiếp theo]", "").strip()
                else:
                    base_clause_ref = cl_ref_raw
                    node_key = f"CL_{meta.get('document_id')}_{art_ref}_{base_clause_ref}"
                    if is_table or node_key in seen_nodes:
                        # Nếu là bảng biểu hoặc Nút Khoản đã có text -> Ép thành Chunk con
                        is_chunk_leaf = True
                    else:
                        # Kịch bản B: Khoản là lá (Khoản nguyên vẹn)
                        is_clause_leaf = True
                        seen_nodes.add(node_key)
        else:
            # Nếu Text rỗng rải rác ngoài Điều (ví dụ: Lời nói đầu văn bản)
            is_chunk_leaf = True

        params.append({
            "doc_id": meta.get("document_id"),
            "doc_num": meta.get("document_number", "N/A"),
            "title": meta.get("title", ""),
            "l_type": meta.get("legal_type", "N/A"),
            "p_date": p_date,
            "year": year,
            "url": meta.get("url", ""),
            "auth_name": meta.get("issuing_authority"),
            "signer_name": meta.get("signer_name"),
            "signer_id": meta.get("signer_id"),
            "eff_date": meta.get("effective_date", ""),
            "doc_status": meta.get("doc_status", "Còn hiệu lực"),
            "doc_toc": meta.get("document_toc", ""),
            "sectors": meta.get("legal_sectors", []),
            "refs": meta.get("legal_basis_refs", []),
            "amended_refs": meta.get("amended_refs", []),
            "replaced_refs": meta.get("replaced_refs", []),
            "repealed_refs": meta.get("repealed_refs", []),
            
            # Thông tin Phân cấp và Lá
            "chunk_id": chunk.get("chunk_id"), # UUID từ Qdrant
            "chunk_idx": meta.get("chunk_index"),
            "chap_ref": meta.get("chapter_ref"),
            "art_ref": art_ref,
            "base_cl_ref": base_clause_ref,
            "raw_cl_ref": cl_ref_raw,
            
            # Cờ lá
            "is_art_leaf": is_article_leaf,
            "is_cl_leaf": is_clause_leaf,
            "is_chk_leaf": is_chunk_leaf,
            
            "is_table": meta.get("is_table", False),
            "ref_cit": meta.get("reference_citation", ""),
            "text": chunk.get("chunk_text", "")
        })

    # 2. CYPHER QUERY: Cấu trúc Đồ thị Động (Dynamic Tree)
    query = """
    UNWIND $batch AS row

    // A. Merge Document chính
    MERGE (d:Document {id: row.doc_id})
    SET d.document_number = row.doc_num,
        d.title = row.title,
        d.promulgation_date = row.p_date,
        d.effective_date = row.eff_date,
        d.year = row.year,
        d.url = row.url,
        d.doc_status = row.doc_status,
        d.document_toc = row.doc_toc,
        d.is_full_text = true

    // B. Merge Authority
    FOREACH (auth IN CASE WHEN row.auth_name IS NOT NULL AND row.auth_name <> 'N/A' THEN [1] ELSE [] END |
        MERGE (a:Authority {name: row.auth_name})
        MERGE (a)-[:ISSUED]->(d)
    )

    // C. Merge Signer
    FOREACH (signer IN CASE WHEN row.signer_name IS NOT NULL AND row.signer_name <> '' THEN [1] ELSE [] END |
        MERGE (s:Signer {name: row.signer_name})
        SET s.signer_id = CASE WHEN row.signer_id IS NOT NULL THEN row.signer_id ELSE s.signer_id END
        MERGE (s)-[:SIGNED]->(d)
    )

    // D. Merge LegalType
    FOREACH (ltype IN CASE WHEN row.l_type IS NOT NULL AND row.l_type <> 'N/A' THEN [1] ELSE [] END |
        MERGE (lt:LegalType {name: row.l_type})
        MERGE (d)-[:HAS_TYPE]->(lt)
    )

    // F. Merge Sectors
    FOREACH (sec_name IN row.sectors |
        MERGE (sec:Sector {name: sec_name})
        MERGE (d)-[:HAS_SECTOR]->(sec)
    )

    // ==========================================
    // E. XÂY DỰNG CẤU TRÚC ĐỒ THỊ ĐỘNG (DYNAMIC TREE)
    // ==========================================
    
    // Nhánh 1: XỬ LÝ ĐIỀU (ARTICLE)
    FOREACH (_o IN CASE WHEN row.art_ref IS NOT NULL THEN [1] ELSE [] END |
        MERGE (art:Article {id: row.doc_id + '_' + row.art_ref})
        ON CREATE SET art.name = row.art_ref, art.chapter_ref = row.chap_ref
        MERGE (art)-[:BELONGS_TO]->(d)
        
        // Nếu Điều là cấp độ lá (Không có khoản) -> Gán Vector ID và Text trực tiếp vào Article
        FOREACH (_leaf IN CASE WHEN row.is_art_leaf THEN [1] ELSE [] END |
            SET art.qdrant_id = row.chunk_id,
                art.text = row.text,
                art.is_table = row.is_table,
                art.reference_citation = row.ref_cit
        )
    )

    // Nhánh 2: XỬ LÝ KHOẢN (CLAUSE)
    FOREACH (_o IN CASE WHEN row.base_cl_ref IS NOT NULL THEN [1] ELSE [] END |
        MERGE (art:Article {id: row.doc_id + '_' + row.art_ref})
        MERGE (cl:Clause {id: row.doc_id + '_' + row.art_ref + '_' + row.base_cl_ref})
        ON CREATE SET cl.name = row.base_cl_ref
        MERGE (cl)-[:PART_OF]->(art)
        
        // Nếu Khoản là cấp độ lá (Khoản nguyên vẹn) -> Gán Vector ID và Text trực tiếp vào Clause
        FOREACH (_leaf IN CASE WHEN row.is_cl_leaf THEN [1] ELSE [] END |
            SET cl.qdrant_id = row.chunk_id,
                cl.text = row.text,
                cl.is_table = row.is_table,
                cl.reference_citation = row.ref_cit
        )
    )

    // Nhánh 3: XỬ LÝ CHUNK MẢNH (SPLIT CHUNK)
    // Trường hợp này xảy ra khi một Khoản bị tách làm nhiều mảnh, HOẮC văn bản không có cấu trúc Điều.
    FOREACH (_o IN CASE WHEN row.is_chk_leaf THEN [1] ELSE [] END |
        MERGE (c:Chunk {id: row.chunk_id})
        SET c.chunk_index = row.chunk_idx,
            c.text = row.text,
            c.is_table = row.is_table,
            c.reference_citation = row.ref_cit,
            c.qdrant_id = row.chunk_id
            
        // Nếu Chunk là con của một Khoản bị vỡ
        FOREACH (_c IN CASE WHEN row.base_cl_ref IS NOT NULL THEN [1] ELSE [] END |
            MERGE (cl:Clause {id: row.doc_id + '_' + row.art_ref + '_' + row.base_cl_ref})
            MERGE (c)-[:PART_OF]->(cl)
        )
        
        // Nếu Chunk là con trực tiếp của một Điều (Điều không có Khoản nhưng bị vỡ nối tiếp hoặc chứa Bảng biểu)
        FOREACH (_a IN CASE WHEN row.base_cl_ref IS NULL AND row.art_ref IS NOT NULL THEN [1] ELSE [] END |
            MERGE (art:Article {id: row.doc_id + '_' + row.art_ref})
            MERGE (c)-[:PART_OF]->(art)
        )
        
        // Nếu Chunk mồ côi (Không thuộc Điều khoản nào)
        FOREACH (_orphan IN CASE WHEN row.art_ref IS NULL THEN [1] ELSE [] END |
            MERGE (c)-[:BELONGS_TO]->(d)
        )
    )

    // ==========================================
    // QUẢN LÝ XUNG ĐỘT (CĂN CỨ VÀ HIỆU LỰC)
    // ==========================================
    
    // G. CĂN CỨ PHÁP LÝ (BASED_ON)
    FOREACH (ref IN row.refs |
        FOREACH (_o IN CASE WHEN ref.doc_number IS NOT NULL AND ref.doc_number <> '' AND ref.doc_number <> 'unknown' THEN [1] ELSE [] END |
            MERGE (p:Document {document_number: ref.doc_number})
            ON CREATE SET p.id = 'REF_' + ref.doc_number, p.is_full_text = false
            
            // CỰC KỲ QUAN TRỌNG: Nếu tra cứu HF thất bại (title rỗng), dùng title từ Regex
            SET p.title = COALESCE(p.title, ref.doc_title),
                p.year = COALESCE(p.year, ref.doc_year)
            
            MERGE (d)-[:BASED_ON]->(p)
        )
    )
    
    // H. SỬA ĐỔI, BỔ SUNG (AMENDS)
    FOREACH (ref IN row.amended_refs |
        FOREACH (_o IN CASE WHEN ref.doc_number IS NOT NULL AND ref.doc_number <> '' AND ref.doc_number <> 'unknown' THEN [1] ELSE [] END |
            MERGE (p:Document {document_number: ref.doc_number})
            ON CREATE SET p.id = 'REF_' + ref.doc_number, p.is_full_text = false
            MERGE (d)-[r:AMENDS {\n
                target_article: COALESCE(ref.target_article, 'TOAN_BO'),\n
                target_clause: COALESCE(ref.target_clause, 'TOAN_BO')\n
            }]->(p)\n
            \n
            SET r.is_entire_doc = ref.is_entire_doc,\n
                r.old_text = ref.old_text,\n
                r.new_text = ref.new_text\n
        )
    )
    
    // I. THAY THẾ (REPLACES)
    FOREACH (ref IN row.replaced_refs |
        FOREACH (_o IN CASE WHEN ref.doc_number IS NOT NULL AND ref.doc_number <> '' AND ref.doc_number <> 'unknown' THEN [1] ELSE [] END |
            MERGE (p:Document {document_number: ref.doc_number})
            ON CREATE SET p.id = 'REF_' + ref.doc_number, p.is_full_text = false
            
            MERGE (d)-[r:REPLACES {
                target_article: COALESCE(ref.target_article, 'TOAN_BO'),
                target_clause: COALESCE(ref.target_clause, 'TOAN_BO')
            }]->(p)
            
            SET r.is_entire_doc = ref.is_entire_doc,
                r.old_text = ref.old_text,
                r.new_text = ref.new_text
            
            // 🟢 LOGIC MỚI: Đánh dấu "Hết hiệu lực" nếu bị thay thế toàn bộ
            FOREACH (_expired IN CASE WHEN ref.is_entire_doc = true THEN [1] ELSE [] END |
                SET p.doc_status = 'Hết hiệu lực'
            )
        )
    )
    
    // J. BÃI BỎ (REPEALS)
    FOREACH (ref IN row.repealed_refs |
        FOREACH (_o IN CASE WHEN ref.doc_number IS NOT NULL AND ref.doc_number <> '' AND ref.doc_number <> 'unknown' THEN [1] ELSE [] END |
            MERGE (p:Document {document_number: ref.doc_number})
            ON CREATE SET p.id = 'REF_' + ref.doc_number, p.is_full_text = false
            
            MERGE (d)-[r:REPEALS {
                target_article: COALESCE(ref.target_article, 'TOAN_BO'),
                target_clause: COALESCE(ref.target_clause, 'TOAN_BO')
            }]->(p)
            
            SET r.is_entire_doc = ref.is_entire_doc,
                r.old_text = ref.old_text,
                r.new_text = ref.new_text
            
            // 🟢 LOGIC MỚI: Đánh dấu "Hết hiệu lực" nếu bị bãi bỏ toàn bộ
            FOREACH (_expired IN CASE WHEN ref.is_entire_doc = true THEN [1] ELSE [] END |
                SET p.doc_status = 'Hết hiệu lực'
            )
        )
    )
    """

    with driver.session() as session:
        session.run(query, batch=params)

def init_neo4j_constraints(driver):
    """Thiết lập các ràng buộc duy nhất để bảo vệ toàn vẹn dữ liệu Graph"""
    queries = [
        # Ràng buộc ID duy nhất
        "CREATE CONSTRAINT document_id_unique IF NOT EXISTS FOR (d:Document) REQUIRE d.id IS UNIQUE;",
        "CREATE CONSTRAINT article_id_unique IF NOT EXISTS FOR (a:Article) REQUIRE a.id IS UNIQUE;",
        "CREATE CONSTRAINT clause_id_unique IF NOT EXISTS FOR (cl:Clause) REQUIRE cl.id IS UNIQUE;",
        "CREATE CONSTRAINT chunk_id_unique IF NOT EXISTS FOR (c:Chunk) REQUIRE c.id IS UNIQUE;",

        # Ràng buộc cho Số hiệu văn bản để làm điểm nối căn cứ (Đổi thành INDEX để hỗ trợ nhiều văn bản "N/A")
        "CREATE INDEX document_number_index IF NOT EXISTS FOR (d:Document) ON (d.document_number);",

        # Ràng buộc cho các thực thể Metadata
        "CREATE CONSTRAINT authority_name_unique IF NOT EXISTS FOR (a:Authority) REQUIRE a.name IS UNIQUE;",
        "CREATE CONSTRAINT sector_name_unique IF NOT EXISTS FOR (s:Sector) REQUIRE s.name IS UNIQUE;",
        "CREATE CONSTRAINT legaltype_name_unique IF NOT EXISTS FOR (lt:LegalType) REQUIRE lt.name IS UNIQUE;"
    ]
    with driver.session() as session:
        for query in queries:
            session.run(query)
    print("🛡️ Hệ thống Constraints (Bao gồm Node Clause) đã sẵn sàng!")

import gc
import json
import time
import uuid
import os
import sys
from contextlib import contextmanager

from qdrant_client import models

# --- BỘ "BỊT MIỆNG" TQDM/LOGS ---
@contextmanager
def suppress_stdout():
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout

# ==========================================
# CẤU HÌNH CHẾ ĐỘ HOẠT ĐỘNG
# ==========================================
PROCEED_UPSERT = True # TẮT (False) để in 1 payload mẫu | BẬT (True) để chạy Full và lưu DB
DOC_BATCH_SIZE = 250
EMBED_BATCH_SIZE = 128
UPSERT_BATCH_SIZE = 512

if not qdrant_client.collection_exists(COLLECTION_NAME):
    raise ValueError(f"Collection {COLLECTION_NAME} chưa tồn tại.")

# ==========================================
# XÂY DỰNG BẢNG TRA CỨU TOÀN CỤC (GLOBAL DOC LOOKUP)
# Dùng để truy xuất chéo nội dung văn bản cũ (old_text) khi phát hiện xung đột
# ==========================================
print("🔍 Đang kết nối và xây dựng Meta Lookup từ TOÀN BỘ CSDL Hugging Face...")

meta_by_docnum_lookup = {}
# Lưu ý: ds_metadata_all phải là split "metadata" đầy đủ từ HF
for r in ds_metadata_all:
    raw_dnum = str(r.get("document_number", "")).strip()
    if raw_dnum and raw_dnum != "N/A":
        # Dùng khóa siêu chuẩn hóa
        key = normalize_doc_key(raw_dnum)
        meta_by_docnum_lookup[key] = r

print(f"✅ Đã nạp {len(meta_by_docnum_lookup):,} văn bản vào bộ nhớ tra cứu metadata.")

print("🔍 Đang xây dựng Global Doc Lookup từ toàn bộ CSDL...")
_meta_all_dict = {str(r["id"]): r for r in ds_metadata_all}
global_doc_lookup = {}
for _row in ds_content_all:
    _did = str(_row["id"])
    _meta = _meta_all_dict.get(_did)
    if _meta:
        _dnum = _meta.get("document_number", "")
        if _dnum and _dnum != "N/A":
            global_doc_lookup[_dnum.strip().upper()] = str(_row.get("text", _row.get("content", "")))
print(f"✅ Global Doc Lookup sẵn sàng: {len(global_doc_lookup):,} văn bản")

all_doc_rows = list(ds_content_selected)
total_docs = len(all_doc_rows)

PIPELINE_STATS = {
    "documents": total_docs,
    "chunks": 0,
    "chunk_seconds": 0.0,
    "embed_seconds": 0.0,
    "point_build_seconds": 0.0,
    "upsert_seconds": 0.0,
}

processed_docs = 0

neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

init_neo4j_constraints(neo4j_driver)

# ==========================================
# VÒNG LẶP CHÍNH
# ==========================================
for start in range(0, total_docs, DOC_BATCH_SIZE):
    # Tối ưu: Nếu chỉ xem mẫu (Preview), ta ép lấy đúng 1 văn bản để nhúng siêu tốc
    current_batch_size = DOC_BATCH_SIZE if PROCEED_UPSERT else 1
    end = min(start + current_batch_size, total_docs)

    if PROCEED_UPSERT:
        print(f"\n🚀 Đang xử lý tài liệu {start + 1} đến {end} / {total_docs}...")
    else:
        print(f"\n🔍 Đang trích xuất văn bản đầu tiên để làm Preview Mẫu...")

    doc_batch = all_doc_rows[start : end]
    if not doc_batch:
        continue

    # 1) Chunking
    t0 = time.perf_counter()
    batch_chunks = []
    for row in doc_batch:
        doc_id = str(row["id"])
        meta = metadata_by_id.get(doc_id)
        if not meta:
            continue
        text_content = str(row.get("text", row.get("content", "")))
        batch_chunks.extend(chunker.process_document(text_content, meta, global_doc_lookup=global_doc_lookup))

    PIPELINE_STATS["chunk_seconds"] += time.perf_counter() - t0

    if not batch_chunks:
        processed_docs += len(doc_batch)
        continue

    # 2) Embedding
    # TỐI ƯU PREVIEW: Nếu chỉ in mẫu 1 chunk, ta chỉ nhúng chunk thứ 3 (index 2) để tránh phần mào đầu rớt của văn bản
    if not PROCEED_UPSERT:
        preview_idx = min(2, len(batch_chunks) - 1)
        texts = [batch_chunks[preview_idx]["chunk_text"]]
    else:
        texts = [c["text_to_embed"] for c in batch_chunks]
        
    t0 = time.perf_counter()

    with suppress_stdout():
        dense_vecs, sparse_vecs = hybrid_encoder.encode_hybrid(texts, batch_size=EMBED_BATCH_SIZE)

    PIPELINE_STATS["embed_seconds"] += time.perf_counter() - t0

    # 3) Build Qdrant Points
    t0 = time.perf_counter()
    points = []

    for i, chunk in enumerate(batch_chunks):
        # LẤY ĐÚNG PAYLOAD CHO TỪNG DB
        q_payload = chunk.get("qdrant_metadata", {})
        n_payload = chunk.get("neo4j_metadata", {})
        qdrant_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk["chunk_id"]))

        # ----------------------------------------------------
        # MODE 2: IN 1 PAYLOAD MẪU DUY NHẤT
        # ----------------------------------------------------
        if not PROCEED_UPSERT:
            if i == preview_idx: # In ra chunk thứ 3 (tránh phần mở đầu rác)
                # Chú ý: Vì mảng vec lúc này chỉ có 1 phần tử (được nhúng từ texts[0])
                sparse_v = sparse_vecs[0]
                sparse_len = len(sparse_v.indices) if hasattr(sparse_v, 'indices') else len(sparse_v)
                print("\n" + "="*80)
                print(f"🔍 [MODE 2] PREVIEW: PAYLOAD MẪU (CHUNK SỐ {i+1})")
                print("="*80)
                print(f"📦 CHUNK ID: {chunk['chunk_id']} | QDRANT UUID: {qdrant_id}")
                print(f"📐 Vector Dense: {len(dense_vecs[0])} dims | Vector Sparse: {sparse_len} points")
                
                print("-" * 80)
                print("🟢 1. METADATA JSON SẼ LƯU VÀO QDRANT (SIÊU NHẸ):")
                print(json.dumps(q_payload, ensure_ascii=False, indent=2))
                
                print("-" * 80)
                print("🔵 2. METADATA JSON SẼ LƯU VÀO NEO4J (FULL BỐI CẢNH):")
                # Lược bỏ document_toc và chunk_text khi in ra để màn hình không bị trôi quá dài
                preview_n_payload = {k: v for k, v in n_payload.items() if k not in ['document_toc', 'chunk_text']}
                print(json.dumps(preview_n_payload, ensure_ascii=False, indent=2))
                print("   ... (Đã ẩn 'document_toc' và 'chunk_text' cho dễ nhìn)")

                print("-" * 80)
                print("📝 TEXT ĐỂ ĐƯA VÀO MODEL NHÚNG:")
                print(chunk["chunk_text"])
                print("="*80)
            continue # Nếu đang Preview, In xong thì bỏ qua, không add vào points

        # ----------------------------------------------------
        # MODE 1: LÀM FULL
        # ----------------------------------------------------
        points.append(
            models.PointStruct(
                id=qdrant_id,
                vector={"dense": dense_vecs[i], "sparse": sparse_vecs[i]},
                payload=q_payload, # CHỈ ĐẨY GÓI QDRANT PAYLOAD LÊN VECTOR DB
            )
        )

    PIPELINE_STATS["point_build_seconds"] += time.perf_counter() - t0

    # 4) Upsert (Chỉ thực thi khi PROCEED_UPSERT = True)
    if PROCEED_UPSERT:
        t0 = time.perf_counter()
        for p_start in range(0, len(points), UPSERT_BATCH_SIZE):
            p_batch = points[p_start : p_start + UPSERT_BATCH_SIZE]
            qdrant_client.upsert(collection_name=COLLECTION_NAME, points=p_batch, wait=True)
        PIPELINE_STATS["upsert_seconds"] += time.perf_counter() - t0

        # 5) GraphDB Push (Neo4j)
        t0_n = time.perf_counter()
        print(f"Đang push {len(batch_chunks)} chunks vào Neo4j")
        # Hàm build_neo4j sẽ tự động bóc neo4j_metadata ra từ batch_chunks
        build_neo4j(neo4j_driver, batch_chunks)
        print(f"✅ Đã push {len(batch_chunks)} chunks vào Neo4j ({time.perf_counter()-t0_n:.2f}s)")

        PIPELINE_STATS["chunks"] += len(batch_chunks)
        processed_docs += len(doc_batch)
        print(f"✅ Xong batch! Lũy kế: {PIPELINE_STATS['chunks']:,} chunks.")

    # Dọn dẹp RAM
    del batch_chunks, texts, dense_vecs, sparse_vecs, points
    gc.collect()

    # Nếu đang ở chế độ Preview, thoát ngay lập tức sau 1 văn bản đầu tiên
    if not PROCEED_UPSERT:
        print("\n⚠️ Đã in xong 1 bản Preview. Dừng tiến trình để bạn kiểm tra!")
        break

neo4j_driver.close()
print("\n" + "="*40)
print("🏁 TIẾN TRÌNH HOÀN TẤT")
print(f"Chế độ Upsert (Làm full): {'BẬT (Đã lưu DB)' if PROCEED_UPSERT else 'TẮT (Chỉ xem mẫu)'}")
if PROCEED_UPSERT:
    print(json.dumps(PIPELINE_STATS, ensure_ascii=False, indent=2))

🔍 Đang kết nối và xây dựng Meta Lookup từ TOÀN BỘ CSDL Hugging Face...
✅ Đã nạp 238,206 văn bản vào bộ nhớ tra cứu metadata.
🔍 Đang xây dựng Global Doc Lookup từ toàn bộ CSDL...
✅ Global Doc Lookup sẵn sàng: 240,119 văn bản
🛡️ Hệ thống Constraints (Bao gồm Node Clause) đã sẵn sàng!

🚀 Đang xử lý tài liệu 1 đến 100 / 100...


Chunks: 100%|██████████| 2/2 [01:08<00:00, 34.36s/it]


Đang push 3352 chunks vào Neo4j
✅ Đã push 3352 chunks vào Neo4j (6.77s)
✅ Xong batch! Lũy kế: 3,352 chunks.

🏁 TIẾN TRÌNH HOÀN TẤT
Chế độ Upsert (Làm full): BẬT (Đã lưu DB)
{
  "documents": 100,
  "chunks": 3352,
  "chunk_seconds": 1.200166929000261,
  "embed_seconds": 69.16588672299986,
  "point_build_seconds": 0.15356312099993374,
  "upsert_seconds": 9.758296255999994
}


In [49]:
# @title Cell Test Nhanh: In Breadcrumb, TOC và Chunk Text của 1 văn bản Y tế
def test_quick_print_medical():
    try:
        # Lấy 1 văn bản bất kỳ trong tập dữ liệu (vốn đã được lọc theo Thể thao - Y tế)
        if not metadata_by_id:
            print("⚠️ Không có dữ liệu metadata_by_id.")
            return
            
        # Lấy ID của văn bản đầu tiên
        target_doc_id = list(metadata_by_id.keys())[0]
        target_meta = metadata_by_id[target_doc_id]
        target_number = target_meta.get('document_number')
            
        content_dict = content_by_id.get(target_doc_id, {})
        target_content = content_dict.get('text', content_dict.get('content', ''))

        print(f"🔍 VĂN BẢN TEST (Y TẾ): {target_number} - {target_meta.get('title')}\n")
        print("=" * 80)
        
        test_chunks = chunker.process_document(target_content, target_meta)
        
        if test_chunks:
            # In TOC 1 lần vì metadata chunk nào cũng lưu bản sao của TOC này
            first_meta = test_chunks[0].get('neo4j_metadata', {})
            print("📑 DOCUMENT TOC (MỤC LỤC):")
            print(first_meta.get('document_toc', 'Không có TOC'))
            print("=" * 80)
            
        for i, c in enumerate(test_chunks):
            q_meta = c.get("qdrant_metadata", {})
            n_meta = c.get("neo4j_metadata", {})
            
            print(f"📦 CHUNK INDEX: {i + 1}")
            print(f"🍞 Breadcrumb: {q_meta.get('breadcrumb')}")
            print(f"📝 Chunk Text:\n{n_meta.get('chunk_text', c.get('chunk_text'))}")
            print("-" * 80)
            
            # Chỉ in 5 chunk đầu cho đỡ dài màn hình
            if i >= 4:
                print(f"... (Đã in 5/{len(test_chunks)} chunk. Dừng để tránh trôi màn hình) ...")
                break
                
    except Exception as e:
        print(f"❌ Lỗi: {str(e)}")

# Chạy In thử test
test_quick_print_medical()

🔍 VĂN BẢN TEST (Y TẾ): 3398/QĐ-UBND - Quyết định 3398/QĐ-UBND năm 2025 công bố Danh mục thủ tục hành chính lĩnh vực An toàn thực phẩm, lĩnh vực Quản lý chất lượng nông lâm sản và thủy sản, lĩnh vực Bảo vệ thực vật thuộc thẩm quyền giải quyết của Sở Y tế, Sở Nông Nghiệp và Môi trường, Ủy ban nhân dân cấp xã, Thành phố Đà Nẵng

📑 DOCUMENT TOC (MỤC LỤC):
Điều 1. Công bố kèm theo Quyết định này danh mục thủ tục hành chính thuộc thẩm quyền giải quyết của Sở Y tế,...
Điều 2. Quyết định này có hiệu lực thi hành kể từ ngày ký và bãi bỏ các quyết định sau:
Điều 3. Chánh Văn phòng Ủy ban nhân dân thành phố, Giám đốc Sở Y tế, Thủ trưởng các sở, ban, ngành; Chủ tịch...
📦 CHUNK INDEX: 1
🍞 Breadcrumb: Điều 1 > Khoản 1, 2.
📝 Chunk Text:
Văn bản: 3398/QĐ-UBND - Quyết định 3398/QĐ-UBND năm 2025 công bố Danh mục thủ tục hành chính lĩnh vực An toàn thực phẩm, lĩn...
Lĩnh vực: Bộ máy hành chính, Thể thao - Y tế
Điều khoản: Điều 1 > Khoản 1, 2.
Nội dung:
Điều 1
Công bố kèm theo Quyết định này danh mục thủ 

In [50]:
# # ==========================================
# # TẠO SNAPSHOT VÀ LƯU LÊN GOOGLE DRIVE
# # ==========================================
# import os
# import shutil

# # 1. Kết nối với Google Drive (Chỉ áp dụng khi chạy trên Colab)
# try:
#     from google.colab import drive
#     # Lệnh này sẽ bật một popup yêu cầu bạn cấp quyền truy cập Google Drive
#     drive.mount('/content/drive')
#     DRIVE_DESTINATION_DIR = "/content/drive/MyDrive/"
#     print("✅ Đã kết nối thành công với Google Drive.")
# except ImportError:
#     print("⚠️ Không chạy trên môi trường Google Colab. Snapshot sẽ được lưu ở thư mục hiện tại.")
#     DRIVE_DESTINATION_DIR = "./"

# print(f"\\n--- Đang bắt đầu quá trình tạo Snapshot cho collection: {COLLECTION_NAME} ---")

# try:
#     # 2. Ra lệnh cho Qdrant tạo bản đóng gói (Snapshot)
#     snapshot_info = qdrant_client.create_snapshot(collection_name=COLLECTION_NAME)
#     snapshot_name = snapshot_info.name

#     # 3. Đường dẫn vật lý của file snapshot bên trong thư mục ảo của Qdrant
#     source_path = f"{QDRANT_LOCAL_PATH}/snapshots/{COLLECTION_NAME}/{snapshot_name}"

#     # 4. Đường dẫn đích trên Google Drive
#     final_output_path = os.path.join(DRIVE_DESTINATION_DIR, snapshot_name)

#     print(f"✅ Đã tạo xong Snapshot cục bộ: {snapshot_name}")
#     print(f"⏳ Đang sao chép file sang Google Drive... (Có thể mất vài phút vì file nặng vài GB)")

#     # 5. Copy file sang Google Drive
#     shutil.copy2(source_path, final_output_path)

#     print("-" * 60)
#     print(f"🎉 THÀNH CÔNG TỐT ĐẸP!")
#     print(f"📁 File Snapshot đã được lưu an toàn tại: {final_output_path}")
#     print(f"👉 Bạn có thể vào Google Drive cá nhân của mình để kiểm tra và tải file về máy local.")
#     print("-" * 60)

# except Exception as e:
#     print(f"❌ Lỗi trong quá trình tạo hoặc lưu Snapshot: {e}")